# Mirror Terrain: Ay ve Mars'ın Dünya'daki ikizleri

**NASA Space Apps Challenge 2026 · Identify Earth Locations that Analog the Permanent Moon Base Locations and Mars**

Bu not defteri projenin bütün veri hattını baştan sona çalıştırır:

1. Dünya'daki aday alanların yükseklik verisini indirir (NASA NASADEM, 60°K üstü için Copernicus GLO-30)
2. Her alanı 12 km'lik pencerelere bölüp arazi özelliklerini hesaplar (eğim, pürüzlülük, rölyef, hipsometri)
3. İklim (NASA POWER) ve çoraklık (ESA WorldCover) katmanlarını ekler
4. Ay güney kutbu hedeflerini (NASA LRO LOLA) ve Mars rover bölgelerini (NASA MOLA, MOLA + ESA HRSC) okur
5. Her hedef için Dünya pencerelerini **aynı çözünürlükte** karşılaştırıp benzerlik skoru verir
6. Modeli bilinen analoglar ve kontrol alanlarıyla doğrular (ilk-20 isabeti, AUC)

## Nasıl çalıştırılır

1. Menüden **Çalışma zamanı → Tümünü çalıştır**'a bas.
2. Google Drive izni istenince onayla. Dosyalar `MyDrive/MirrorTerrain` klasörüne kaydedilir, oturum kapansa da kaybolmaz.
3. OpenTopography API anahtarı istenince yapıştır. (Daha pratik yol: soldaki 🔑 *Secrets* menüsünde `OPENTOPO_KEY` adıyla kaydet ve "Notebook access" anahtarını aç; o zaman hiç sormaz.)

Daha önce hesaplanan sonuçlar CSV olarak saklanır ve tekrar çalıştırınca **önbellekten yüklenir**; ilk çalıştırma 10-20 dakika, sonrakiler 1-3 dakika sürer. Her şeyi baştan hesaplatmak için Bölüm 1'de `FORCE = True` yap.

> API anahtarı not defterine yazılmaz; bu dosya GitHub'a güvenle yüklenebilir.

## 0. Kurulum

In [ ]:
!pip install -q rasterio pyproj

import os, re, time, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.warp import transform as warp_xy
from rasterio.windows import from_bounds
from scipy import ndimage
from pyproj import CRS, Transformer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

# Google Drive: dosyalar oturum kapansa da kalsın
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/MirrorTerrain"
os.makedirs(f"{WORK}/dem", exist_ok=True)
os.chdir(WORK)
print("Çalışma klasörü:", os.getcwd())

# API anahtarı: önce Colab Secrets, yoksa sor
try:
    from google.colab import userdata
    API_KEY = userdata.get("OPENTOPO_KEY")
except Exception:
    from getpass import getpass
    API_KEY = getpass("OpenTopography API anahtarı: ")

## 1. Ayarlar

Alan listesi, hedefler ve modelin ağırlıkları burada. Yeni bir Dünya adayı eklemek için `SITES` listesine bir satır eklemen yeterli.

- **known:** NASA/ESA'nın kullandığı bilinen analoglar (doğrulama için; model bunları üst sıralara koymalı)
- **turkey:** Türkiye adayları
- **control:** Ay'a ya da Mars'a benzememesi gereken sıradan araziler (doğrulama için; düşük skor almalı)

In [ ]:
FORCE = False          # True: önbellekteki CSV'leri yok say, her şeyi baştan hesapla
RUN_TR_SCAN = True     # Türkiye sistematik taraması (Bölüm 3b)
SCAN_MIN_BARREN = 0.30 # taramada aday sayılmak için en az çoraklık (0,1° hücre düzeyinde)
SCAN_MAX_SITES = 25    # en fazla kaç tarama adayı indirilsin
SCAN_MIN_SEP = 0.3    # adaylar arası (ve mevcut alanlara) en az uzaklık, derece

SITES = [
    # bilinen analoglar (doğrulama)
    ("DeathValley",          36.25, -116.82, "known"),
    ("Atacama_Yungay",      -24.10,  -70.00, "known"),
    ("MeteorCrater",         35.03, -111.02, "known"),
    ("Hawaii_MaunaKea",      19.76, -155.45, "known"),
    ("Iceland_Holuhraun",    64.87,  -16.83, "known"),
    ("Haughton",             75.38,  -89.68, "known"),
    ("Svalbard_Bockfjorden", 79.43,   13.33, "known"),
    # Türkiye adayları
    ("TR_Kula",              38.57,   28.65, "turkey"),
    ("TR_Kapadokya",         38.64,   34.83, "turkey"),
    ("TR_TuzGolu",           38.80,   33.40, "turkey"),
    ("TR_Nemrut",            38.65,   42.23, "turkey"),
    ("TR_Salda",             37.55,   29.68, "turkey"),   # NASA: Jezero'nun mineral analoğu (hidromanyezit + delta)
    # kontrol alanları (benzememeli)
    ("CTRL_Kansas",          38.50,  -98.50, "control"),
    ("CTRL_Amazon",          -3.50,  -60.00, "control"),
    ("CTRL_BlackForest",     48.00,    8.20, "control"),
]

MOON_TARGETS = [                     # yaklaşık koordinatlar (enlem, boylam)
    ("Moon_Shackleton",   -89.66,  129.8),
    ("Moon_deGerlache",   -88.50,  -68.0),
    ("Moon_Malapert",     -86.00,    2.9),
]
MARS_TARGETS = [
    ("Mars_Gusev_Spirit",      -14.60, 175.50),
    ("Mars_Jezero",             18.44,  77.45),
    ("Mars_Gale_Curiosity",     -4.59, 137.44),
    ("Mars_Meridiani_Oppy",     -1.95,  -5.53),
]

# NASA LRO LOLA güney kutbu yükseklik haritası, 40 m/piksel (cloud-optimized GeoTIFF)
LOLA_URL = "https://pgda.gsfc.nasa.gov/data/LOLA_20mpp/LDEM_80S_40MPP_ADJ.TIF"
# Mars: NASA MOLA + ESA HRSC birleşik (200 m) ve NASA MOLA (463 m)
MARS_SOURCES = {
    "HRSC_MOLA_200m": "https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif",
    "MOLA_463m": "https://asc-pds-services.s3.us-west-2.amazonaws.com/wms_basemaps3/Mars/MOLA/mola128_mola64_merge_90Nto90S_SimpleC_clon0_cog.tif",
}
MARS_R = 3396190.0                   # Mars yarıçapı (m)

WIN_M, STEP_M = 12000, 6000          # 12 km pencere, 6 km adım

# Arazi özellikleri
F = ["relief_m", "slope_mean", "slope_p90", "steep_frac", "roughness_m", "hypsometry"]
# Benzerlik ağırlıkları (başlangıç değerleri; çoraklık analogları kontrollerden en iyi ayıran özellik)
W = pd.Series({"relief_m": 1, "slope_mean": 1, "slope_p90": 1, "steep_frac": 1,
               "roughness_m": 1, "hypsometry": 0.5, "w_barren_frac": 2})
W_TERRAIN = W.drop("w_barren_frac")  # doğrulama testi: sadece arazi

LC = ["bare_frac", "snow_ice_frac", "lichen_frac", "veg_frac", "barren_frac"]
# Ölçekleme sabit bir referans gruba göre yapılır: tarama adayları eklense de skorlar ve doğrulama değişmez
REF_GROUPS = ["known", "turkey", "control"]
EXTRA = ["w_barren_frac", "w_bare_frac", "w_veg_frac", "precip_mm_yr", "t_mean_c", "t_daily_range_c"]
MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]

## 2. Fonksiyonlar

Bu hücre sadece fonksiyonları tanımlar, ekrana bir şey yazmaz.

In [ ]:
# ---------- Dünya yükseklik verisi ----------
def utm_epsg(lat, lon):
    zone = int((lon + 180) // 6) + 1
    return f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"

def download(name, lat, lon, half_km=17):
    # NASADEM 60°K'nin üstünü kapsamıyor; orada Copernicus GLO-30
    dlat = half_km / 111.0
    dlon = half_km / (111.0 * np.cos(np.radians(lat)))
    dem = "NASADEM" if -56 <= lat <= 60 else "COP30"
    path = f"dem/{name}_{dem}.tif"
    if not os.path.exists(path):
        url = ("https://portal.opentopography.org/API/globaldem?"
               f"demtype={dem}&south={lat-dlat}&north={lat+dlat}"
               f"&west={lon-dlon}&east={lon+dlon}&outputFormat=GTiff&API_Key={API_KEY}")
        r = requests.get(url, timeout=180)
        r.raise_for_status()
        open(path, "wb").write(r.content)
    return path, dem

def to_utm(path, lat, lon, res=30):
    # derece cinsinden haritayı, alanın UTM bölgesinde res metrelik ızgaraya çevirir
    out = path.replace(".tif", f"_{res}m.tif")
    if os.path.exists(out):
        return out
    dst_crs = utm_epsg(lat, lon)
    with rasterio.open(path) as src:
        tr, w, h = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=res)
        meta = src.meta.copy()
        meta.update(crs=dst_crs, transform=tr, width=w, height=h, dtype="float32", nodata=np.nan)
        with rasterio.open(out, "w", **meta) as dst:
            reproject(rasterio.band(src, 1), rasterio.band(dst, 1),
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=tr, dst_crs=dst_crs, resampling=Resampling.bilinear,
                      src_nodata=src.nodata, dst_nodata=np.nan)
    return out

# ---------- Arazi özellikleri ----------
def features(z, res):
    gy, gx = np.gradient(z, res)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    rough = np.std(z - ndimage.uniform_filter(z, size=15))
    zmin, zmax = np.percentile(z, [1, 99])
    relief = zmax - zmin
    return {
        "relief_m": relief,                                         # toplam yükseklik farkı
        "slope_mean": slope.mean(),                                 # ortalama eğim (derece)
        "slope_p90": np.percentile(slope, 90),                      # en dik %10'un başladığı eğim
        "steep_frac": (slope > 15).mean(),                          # 15°'den dik alan oranı
        "roughness_m": rough,                                       # küçük ölçekli pürüzlülük
        "hypsometry": np.clip((z.mean() - zmin) / max(relief, 1.0), 0, 1),
        "water_frac": ((np.abs(z) < 1) & (slope < 0.5)).mean(),     # deniz seviyesinde ve düz = su
    }, slope

def site_windows(name, lat, lon, group, res=30):
    path, dem = download(name, lat, lon)
    with rasterio.open(to_utm(path, lat, lon, res)) as s:
        z, r_, tr, crs = s.read(1), s.res[0], s.transform, s.crs
    n, st = int(WIN_M / r_), int(STEP_M / r_)
    rows = []
    for i in range(0, z.shape[0] - n + 1, st):
        for j in range(0, z.shape[1] - n + 1, st):
            w = z[i:i + n, j:j + n]
            if np.isnan(w).mean() > 0.01:          # kenardaki boş pencereleri atla
                continue
            w = np.where(np.isnan(w), np.nanmean(w), w)
            f, _ = features(w, r_)
            x, y = tr * (j + n / 2, i + n / 2)      # pencere merkezi
            lo, la = warp_xy(crs, "EPSG:4326", [x], [y])
            rows.append({"site": name, "group": group, "dem": dem,
                         "lat": la[0], "lon": lo[0], **f})
    return rows

# ---------- İklim: NASA POWER ----------
def power_climate(lat, lon):
    url = ("https://power.larc.nasa.gov/api/temporal/climatology/point"
           "?parameters=T2M,T2M_MAX,T2M_MIN,PRECTOTCORR&community=RE"
           f"&longitude={lon}&latitude={lat}&format=JSON")
    p = requests.get(url, timeout=60).json()["properties"]["parameter"]
    return {
        "precip_mm_yr": p["PRECTOTCORR"]["ANN"] * 365,
        "t_mean_c": p["T2M"]["ANN"],
        "t_daily_range_c": p["T2M_MAX"]["ANN"] - p["T2M_MIN"]["ANN"],
        **{f"t_{m.lower()}": p["T2M"][m] for m in MONTHS},
        **{f"p_{m.lower()}": p["PRECTOTCORR"][m] for m in MONTHS},
    }

# ---------- Çoraklık: ESA WorldCover 2021 (10 m) ----------
def wc_tile(lat, lon):
    la, lo = int(np.floor(lat / 3) * 3), int(np.floor(lon / 3) * 3)
    ns = f"N{la:02d}" if la >= 0 else f"S{-la:02d}"
    ew = f"E{lo:03d}" if lo >= 0 else f"W{-lo:03d}"
    return ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
            f"ESA_WorldCover_10m_2021_v200_{ns}{ew}_Map.tif")

def landcover(lat, lon, half_km=6):
    dlat = half_km / 111.0
    dlon = half_km / (111.0 * np.cos(np.radians(lat)))
    with rasterio.open(wc_tile(lat, lon)) as src:
        win = from_bounds(lon - dlon, lat - dlat, lon + dlon, lat + dlat, src.transform)
        lc = src.read(1, window=win, out_shape=(1000, 1000),
                      resampling=Resampling.nearest, boundless=True, fill_value=0)
    land = (lc > 0) & (lc != 80)                      # boş ve su pikselleri hariç
    frac = lambda codes: np.isin(lc, codes)[land].mean()
    return {"bare_frac": frac([60]), "snow_ice_frac": frac([70]), "lichen_frac": frac([100]),
            "veg_frac": frac([10, 20, 30, 40, 90, 95]), "barren_frac": frac([60, 70, 100])}

# ---------- Farklı çözünürlükteki tabloları eşleştirme ----------
def nearest_join(a, b, cols):
    # a'daki her pencereye, b'de aynı alandaki en yakın pencerenin değerlerini taşır
    out = []
    for _, r in a.iterrows():
        s = b[b["site"] == r["site"]]
        d = (s["lat"] - r["lat"]) ** 2 + ((s["lon"] - r["lon"]) * np.cos(np.radians(r["lat"]))) ** 2
        out.append(s.loc[d.idxmin(), cols])
    return pd.DataFrame(out, index=a.index)

def earth_at(res, base):
    # Dünya alanlarını hedefin çözünürlüğünde işler; iklim ve çoraklığı 30 m tablosundan taşır.
    # Önbellekte olmayan (yeni eklenen) alanları hesaplayıp tabloya ekler.
    path = f"features_earth_{res}m.csv"
    e = pd.read_csv(path) if (os.path.exists(path) and not FORCE) else pd.DataFrame(columns=["site"])
    missing = [s for s in SITES if s[0] not in set(e["site"]) and s[0] in set(base["site"])]
    if missing:
        rows = []
        for name, lat, lon, group in missing:
            try:
                rows += site_windows(name, lat, lon, group, res=res)
            except Exception as err:
                print(f"✗ {name} ({res} m): {err}")
        if rows:
            new = pd.DataFrame(rows)
            new = new[new["water_frac"] < 0.2].reset_index(drop=True)
            new = new.join(nearest_join(new, base, EXTRA))
            e = pd.concat([e, new], ignore_index=True)
            e.to_csv(path, index=False)
            print(f"{res} m tablosuna eklendi:", ", ".join(sorted(set(new["site"]))))
    e = e[e["site"].isin([s[0] for s in SITES])].copy()
    e["dem_flat"] = e["roughness_m"] < 0.05           # DEM'in düzleştirdiği göl yüzeyleri
    return e

# ---------- Ay ve Mars pencereleri ----------
# Uzak sunucular (ör. NASA PGDA) yoğunlukta isteği geçici olarak reddedebiliyor (HTTP 403).
# Bu yüzden her hedef penceresi bir kez okunup Drive'a kaydedilir; hata olursa birkaç kez beklenip yeniden denenir.
os.environ.setdefault("GDAL_HTTP_MAX_RETRY", "5")
os.environ.setdefault("GDAL_HTTP_RETRY_DELAY", "10")

def cached_window(key, fn, *args):
    path = os.path.join("dem", "cache", re.sub(r"[^A-Za-z0-9_.-]", "_", key) + ".npz")
    if os.path.exists(path) and not FORCE:
        d = np.load(path)
        return d["z"], float(d["res"])
    last = None
    for attempt in range(4):
        try:
            z, res = fn(*args)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            np.savez_compressed(path, z=z, res=res)
            return z, res
        except Exception as err:
            last = err
            wait = 30 * (attempt + 1)
            print(f"   {key}: okuma başarısız ({str(err)[:70]}), {wait} sn sonra yeniden denenecek...")
            time.sleep(wait)
    raise RuntimeError(f"{key} okunamadı: {last}")

def moon_window(lat, lon, size_m=WIN_M):
    return cached_window(f"moon_{lat}_{lon}_{size_m}", _moon_window_remote, lat, lon, size_m)

def mars_window(url, lat, lon, size_m=WIN_M):
    return cached_window(f"mars_{os.path.basename(url)}_{lat}_{lon}_{size_m}", _mars_window_remote, url, lat, lon, size_m)

def _moon_window_remote(lat, lon, size_m=WIN_M):
    with rasterio.open(LOLA_URL) as src:
        crs = CRS.from_wkt(src.crs.to_wkt())
        tf = Transformer.from_crs(crs.geodetic_crs, crs, always_xy=True)
        x, y = tf.transform(lon, lat)
        h = size_m / 2
        win = from_bounds(x - h, y - h, x + h, y + h, src.transform)
        z = src.read(1, window=win).astype(float)
        if src.nodata is not None:
            z[z == src.nodata] = np.nan
        return z, abs(src.res[0])

def _mars_window_remote(url, lat, lon, size_m=WIN_M):
    with rasterio.open(url) as src:
        crs = CRS.from_wkt(src.crs.to_wkt())
        h = size_m / 2
        if crs.is_geographic:                         # derece cinsinden harita
            x, y = lon, lat
            hy = np.degrees(h / MARS_R)
            hx = hy / np.cos(np.radians(lat))
        else:                                         # metre cinsinden harita
            tf = Transformer.from_crs(crs.geodetic_crs, crs, always_xy=True)
            x, y = tf.transform(lon, lat)
            hy, hx = h, h / np.cos(np.radians(lat))
        win = from_bounds(x - hx, y - hy, x + hx, y + hy, src.transform)
        z = src.read(1, window=win).astype(float)
        if src.nodata is not None:
            z[z == src.nodata] = np.nan
    # Güvenlik ağı: pencerenin ortancasından 1.500 m'den (ve 10 çeyrekler-arası açıklıktan) fazla sapan,
    # pencerenin %5'inden azını oluşturan kopuk piksel grubu gerçek arazi değil, veri hatasıdır.
    med = np.nanmedian(z)
    q1, q3 = np.nanpercentile(z, [25, 75])
    bad = np.abs(z - med) > max(1500, 10 * (q3 - q1))
    if 0 < bad.mean() < 0.05:
        print(f"   veri hatası temizlendi: {int(bad.sum())} piksel")
        z[bad] = np.nan
    # doğu-batı ölçeğini düzelt (kare piksel). mode="nearest" şart: varsayılan mod bazı boyutlarda
    # (ör. Jezero'nun 60x63 penceresi) kenara 0 m'lik sahte bir sütun ekliyordu.
    z = ndimage.zoom(z, (1, z.shape[0] / z.shape[1]), order=1, mode="nearest")
    return z, size_m / z.shape[0]

# ---------- Benzerlik, sıralama, doğrulama ----------
def rank(cand, tg, w):
    # her hedef için her Dünya penceresine 0-100 arası benzerlik skoru (100 = birebir aynı)
    FS = list(w.index)
    ref = cand[cand["group"].isin(REF_GROUPS)]
    sc = RobustScaler().fit(pd.concat([ref[FS], tg[FS]]))
    C = sc.transform(cand[FS])
    out = []
    for _, t in tg.iterrows():
        tv = sc.transform(t[FS].to_frame().T.astype(float))
        d = np.sqrt((((C - tv) * w.values) ** 2).sum(axis=1) / w.sum())
        r = cand[["site", "group", "lat", "lon"]].copy()
        r["target"], r["score"] = t["site"], 100 * np.exp(-d)
        out.append(r)
    return pd.concat(out, ignore_index=True)

def show(res, n=5):
    for tname, g in res.groupby("target"):
        print(f"\n=== {tname}: en benzer {n} Dünya penceresi ===")
        print(g.nlargest(n, "score")[["site", "group", "lat", "lon", "score"]]
              .round(2).to_string(index=False))

def validate(res, k=20):
    # ilk-k isabeti: en üstteki k pencerenin yüzde kaçı bilinen analog
    # AUC: rastgele bir bilinen analog penceresinin rastgele bir kontrol penceresinden yüksek skor alma olasılığı
    rows = []
    for t, g in res.groupby("target"):
        g = g[g["group"].isin(["known", "control"])]
        top = g.nlargest(k, "score")
        rows.append({"target": t,
                     f"ilk{k}_bilinen_%": 100 * (top["group"] == "known").mean(),
                     "AUC": roc_auc_score((g["group"] == "known").astype(int), g["score"])})
    return pd.DataFrame(rows).round(2)

print("Fonksiyonlar hazır.")

## 3. Dünya alanları (30 m): arazi, iklim, çoraklık

İlk çalıştırmada 14 alanı indirir, NASA POWER'dan iklimi ve ESA WorldCover'dan her pencerenin çoraklığını çeker (5-10 dakika). Sonraki çalıştırmalarda önbellekten yükler.

In [ ]:
def build_earth30(sites):
    # verilen alanlar için: 30 m pencereler + su filtresi + NASA POWER iklimi + pencere çoraklığı
    rows = []
    for name, lat, lon, group in sites:
        try:
            r = site_windows(name, lat, lon, group, res=30)
            rows += r
            print(f"✓ {name}: {len(r)} pencere")
        except Exception as e:
            print(f"✗ {name}: {e}")
        time.sleep(1)
    if not rows:
        return pd.DataFrame(columns=["site"])
    e = pd.DataFrame(rows)
    before = len(e)
    e = e[e["water_frac"] < 0.2].reset_index(drop=True)
    print(f"Su ağırlıklı {before - len(e)} pencere çıkarıldı")

    clim = pd.read_csv("climate_earth.csv") if os.path.exists("climate_earth.csv") else pd.DataFrame(columns=["site"])
    todo = [s for s in sites if s[0] not in set(clim["site"])]
    if todo:
        add = []
        for name, lat, lon, group in todo:
            add.append({"site": name, **power_climate(lat, lon)})
            time.sleep(1)
        clim = pd.concat([clim, pd.DataFrame(add)], ignore_index=True)
        clim.to_csv("climate_earth.csv", index=False)
    e = e.merge(clim, on="site", how="left")

    lcw = []
    for k, (_, r) in enumerate(e.iterrows(), 1):
        try:
            d = landcover(r["lat"], r["lon"], half_km=6)
        except Exception:
            d = {c: np.nan for c in LC}
        lcw.append(d)
        if k % 20 == 0:
            print(f"çoraklık: {k}/{len(e)} pencere")
    return e.join(pd.DataFrame(lcw, index=e.index).add_prefix("w_"))

path = "features_earth.csv"
if os.path.exists(path) and not FORCE:
    earth = pd.read_csv(path).drop_duplicates(["site", "lat", "lon"]).reset_index(drop=True)  # eski sürümlerdeki tekrarları temizle
    earth.to_csv(path, index=False)
    print("Önbellekten yüklendi:", path, "|", len(earth), "pencere")
else:
    earth = pd.DataFrame(columns=["site"])
missing = [s for s in SITES if s[0] not in set(earth["site"])]
if missing:
    print("Yeni alanlar işleniyor:", ", ".join(m[0] for m in missing))
    earth = pd.concat([earth, build_earth30(missing)], ignore_index=True)
    earth.to_csv(path, index=False)
earth = earth[earth["site"].isin([s[0] for s in SITES])].reset_index(drop=True)

earth["dem_flat"] = earth["roughness_m"] < 0.05
print("Toplam pencere:", len(earth), "| arazi verisi güvenilmez (düzleştirilmiş göl):", int(earth["dem_flat"].sum()))

summary = earth.groupby(["group", "site"]).agg(
    pencere=("site", "size"), egim=("slope_mean", "mean"), puruzluluk=("roughness_m", "mean"),
    yagis_mm=("precip_mm_yr", "first"), coraklik_ort=("w_barren_frac", "mean"),
    coraklik_max=("w_barren_frac", "max"))
print(summary.round(2))

## 3b. Türkiye sistematik taraması: keşfedilmemiş analog adayları

Elle seçtiğimiz Türkiye alanlarının çoğu bitki örtülü çıktı. Bu bölüm tahmin yerine **bütün Türkiye'yi** tarar:

1. **Ucuz ön eleme:** ESA WorldCover haritasından Türkiye'yi 0,1°'lik (~11 km) hücrelere bölüp her hücrenin çoraklığını hesaplar. Ülke sınırı Natural Earth verisinden alınır.
2. **Aday seçimi:** Çoraklığı en az `SCAN_MIN_BARREN` olan hücreleri birbirine 0,3°'den yakın olmayacak şekilde seçer (mevcut alanlara yakın olanları atlar), en fazla `SCAN_MAX_SITES` aday.
3. **Ayrıntılı analiz:** Adaylar `scan` grubu olarak alan listesine eklenir; yükseklik verisi indirilip diğer alanlarla aynı şekilde işlenir. Sonraki bütün bölümler (sıralama, ikiz takımı) adayları otomatik içerir.

İlk çalıştırmada 14 WorldCover parçası okunduğu ve adaylar indirildiği için 10-20 dakika sürebilir; sonra önbellekten gelir.

In [ ]:
if RUN_TR_SCAN:
    import geopandas as gpd
    from shapely.geometry import Point
    from shapely.prepared import prep

    # Türkiye sınırı (Natural Earth 1:50m)
    NE_URLS = ["https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson",
               "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip"]
    world = None
    for u in NE_URLS:
        try:
            world = gpd.read_file(u)
            break
        except Exception as err:
            print("Sınır verisi okunamadı:", u, err)
    iso_col = "ISO_A3" if "ISO_A3" in world.columns else "iso_a3"
    tr_geom = world[world[iso_col] == "TUR"].geometry
    tr_poly = prep(tr_geom.union_all() if hasattr(tr_geom, "union_all") else tr_geom.unary_union)

    # 1) WorldCover ön eleme: 3°'lik parçaları ~333 m'de okuyup 0,1°'lik hücrelere böl
    grid_path = "scan_grid_turkey.csv"
    if os.path.exists(grid_path) and not FORCE:
        grid = pd.read_csv(grid_path)
        print("Tarama ızgarası önbellekten yüklendi:", len(grid), "hücre")
    else:
        rows = []
        for la0 in (36, 39):
            for lo0 in range(24, 45, 3):
                try:
                    with rasterio.open(wc_tile(la0 + 1.5, lo0 + 1.5)) as src:
                        lc = src.read(1, out_shape=(900, 900), resampling=Resampling.nearest)
                        b = src.bounds
                except Exception as err:
                    print(f"✗ parça {la0}/{lo0}: {err}")
                    continue
                k, cells = 30, 30                      # 900 px / 30 = 30 px'lik (0,1°) hücreler
                for i in range(cells):
                    for j in range(cells):
                        lat = b.top - (i + 0.5) * (b.top - b.bottom) / cells
                        lon = b.left + (j + 0.5) * (b.right - b.left) / cells
                        if not tr_poly.contains(Point(lon, lat)):
                            continue
                        blk = lc[i * k:(i + 1) * k, j * k:(j + 1) * k]
                        land = (blk > 0) & (blk != 80)
                        if land.mean() < 0.5:
                            continue
                        frac = lambda codes: np.isin(blk, codes)[land].mean()
                        rows.append({"lat": round(lat, 3), "lon": round(lon, 3),
                                     "barren": frac([60, 70, 100]), "bare": frac([60]),
                                     "snow_ice": frac([70]), "veg": frac([10, 20, 30, 40, 90, 95])})
                print(f"✓ parça {la0}°K {lo0}°D")
        grid = pd.DataFrame(rows)
        grid.to_csv(grid_path, index=False)
    print("Türkiye hücre sayısı:", len(grid))
    print("Çoraklık dağılımı (hücre sayısı):")
    print(pd.cut(grid["barren"], [0, 0.1, 0.3, 0.5, 0.8, 1.0], include_lowest=True).value_counts().sort_index())

    # 2) Aday seçimi: çorak hücreler, birbirinden ve mevcut alanlardan en az 0,3° uzak
    taken = [(la, lo) for _, la, lo, _ in SITES]
    scan_sites = []
    for _, r in grid[grid["barren"] >= SCAN_MIN_BARREN].sort_values("barren", ascending=False).iterrows():
        if all((r["lat"] - a) ** 2 + (r["lon"] - b) ** 2 > SCAN_MIN_SEP ** 2 for a, b in taken):
            name = f"SCAN_{r['lat']:.2f}N_{r['lon']:.2f}E"
            scan_sites.append((name, float(r["lat"]), float(r["lon"]), "scan"))
            taken.append((r["lat"], r["lon"]))
        if len(scan_sites) >= SCAN_MAX_SITES:
            break
    print(f"\n{len(scan_sites)} tarama adayı seçildi (çoraklık ≥ {SCAN_MIN_BARREN}):")
    for n, la, lo, _ in scan_sites:
        print(f"  {n}  çoraklık {grid.loc[(grid.lat.round(2) == round(la, 2)) & (grid.lon.round(2) == round(lo, 2)), 'barren'].max():.2f}")

    # 3) Adayları alan listesine ekle ve 30 m'de işle (önbellekte olmayanlar)
    SITES = [s for s in SITES if s[3] != "scan"] + scan_sites
    earth_all = pd.read_csv("features_earth.csv").drop_duplicates(["site", "lat", "lon"])   # önbelleğin tamamı (tarama dahil)
    missing = [s for s in SITES if s[0] not in set(earth_all["site"])]
    if missing:
        print("\nAdaylar işleniyor:", len(missing))
        earth_all = pd.concat([earth_all, build_earth30(missing)], ignore_index=True).drop_duplicates(["site", "lat", "lon"])
        earth_all.to_csv("features_earth.csv", index=False)
    earth = earth_all[earth_all["site"].isin([s[0] for s in SITES])].reset_index(drop=True)
    earth["dem_flat"] = earth["roughness_m"] < 0.05
    print("Toplam pencere (tarama dahil):", len(earth))

    # Türkiye haritası
    fig, ax = plt.subplots(figsize=(13, 6))
    sc_ = ax.scatter(grid["lon"], grid["lat"], c=grid["barren"], cmap="YlOrBr", s=9, vmin=0, vmax=1, marker="s")
    plt.colorbar(sc_, ax=ax, label="çoraklık (WorldCover)")
    for n, la, lo, _ in scan_sites:
        ax.plot(lo, la, "o", mfc="none", mec="tab:blue", ms=10)
    for n, la, lo, g in SITES:
        if g == "turkey":
            ax.plot(lo, la, "^", color="tab:green", ms=9)
            ax.annotate(n.replace("TR_", ""), (lo, la), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_title("Türkiye taraması: çoraklık (renk), tarama adayları (mavi daire), elle seçilen alanlar (yeşil üçgen)")
    ax.set_xlabel("boylam"); ax.set_ylabel("enlem"); ax.set_aspect(1.25)
    plt.tight_layout(); plt.savefig("fig_turkey_scan.png", dpi=150); plt.show()

## 4. Ay güney kutbu hedefleri (NASA LRO LOLA, 40 m)

Dünya alanları da **40 m'de** yeniden işlenir; böylece her karşılaştırma aynı çözünürlükte yapılır.

In [ ]:
moon_rows, moon_wins = [], {}
for name, lat, lon in MOON_TARGETS:
    z, r = moon_window(lat, lon)
    print(f"{name}: {z.shape}, piksel {r:.0f} m, boş piksel: {np.isnan(z).mean():.1%}")
    z = np.where(np.isnan(z), np.nanmean(z), z)
    f, s = features(z, r)
    moon_rows.append({"site": name, "body": "Moon", "lat": lat, "lon": lon,
                      "res_m": r, "w_barren_frac": 1.0, **f})
    moon_wins[name] = (z, s)
moon_t = pd.DataFrame(moon_rows)
moon_t.to_csv("features_targets_moon.csv", index=False)
print(moon_t[["site"] + F].round(2).to_string(index=False))

R_MOON = int(round(moon_t["res_m"].iloc[0]))
earth_moon = earth_at(R_MOON, earth)
cand_moon = earth_moon[~earth_moon["dem_flat"]].copy()
print(f"\nDünya pencere sayısı ({R_MOON} m):", len(cand_moon))

ranks_moon = rank(cand_moon, moon_t, W)
ranks_moon_terrain = rank(cand_moon, moon_t, W_TERRAIN)
ranks_moon.to_csv("ranks_moon.csv", index=False)
show(ranks_moon)

fig, ax = plt.subplots(2, len(moon_wins), figsize=(5 * len(moon_wins), 9))
for k, (name, (z, s)) in enumerate(moon_wins.items()):
    ax[0, k].imshow(z, cmap="gray"); ax[0, k].set_title(name)
    ax[1, k].imshow(s, cmap="magma", vmax=40); ax[1, k].set_title("eğim (derece)")
plt.tight_layout(); plt.savefig("fig_moon_targets.png", dpi=150); plt.show()

## 5. Mars rover bölgeleri (NASA MOLA 463 m ve MOLA + ESA HRSC 200 m)

Her hedef iki kaynaktan okunur. 200 m'lik veri daha ayrıntılı ama bazı yerlerde hatalı olabilir; bu yüzden iki kaynak karşılaştırılır ve 200 m'lik değeri 463 m'lik değerle çelişen hedefler **şüpheli** işaretlenip 200 m sıralamasından çıkarılır. 200 m'lik dosya satır satır okunduğu için bu hücre birkaç dakika sürebilir.

In [ ]:
mars_all, mars_wins = [], {}
for label, url in MARS_SOURCES.items():
    for name, lat, lon in MARS_TARGETS:
        z, r = mars_window(url, lat, lon)
        print(f"{label} | {name}: {z.shape}, piksel {r:.0f} m, boş: {np.isnan(z).mean():.1%}")
        z = np.where(np.isnan(z), np.nanmean(z), z)
        f, s = features(z, r)
        mars_all.append({"site": name, "source": label, "body": "Mars", "lat": lat, "lon": lon,
                         "res_m": r, "w_barren_frac": 1.0, **f})
        mars_wins[(label, name)] = (z, s)
mars_all = pd.DataFrame(mars_all)

# Tutarlılık kontrolü: 200 m'lik rölyef 463 m'lik rölyefin 3 katından fazlaysa (ve 500 m'yi aşıyorsa) şüpheli
hr = mars_all[mars_all["source"] == "HRSC_MOLA_200m"].set_index("site")
mo = mars_all[mars_all["source"] == "MOLA_463m"].set_index("site")
check = pd.DataFrame({"relief_200m": hr["relief_m"], "relief_463m": mo["relief_m"],
                      "rough_200m": hr["roughness_m"], "rough_463m": mo["roughness_m"]})
check["supheli"] = (check["relief_200m"] > 3 * check["relief_463m"]) & (check["relief_200m"] > 500)
print("\nTutarlılık kontrolü:")
print(check.round(1))
SUSPECT = set(check.index[check["supheli"]])
mars_all["suspect"] = (mars_all["source"] == "HRSC_MOLA_200m") & mars_all["site"].isin(SUSPECT)
mars_all.to_csv("features_targets_mars.csv", index=False)

mars_ranks = {}
for label in MARS_SOURCES:
    tg = mars_all[(mars_all["source"] == label) & ~mars_all["suspect"]]
    R = int(round(tg["res_m"].iloc[0]))
    e = earth_at(R, earth)
    c = e[~e["dem_flat"]].copy()
    mars_ranks[label] = (rank(c, tg, W), rank(c, tg, W_TERRAIN))
    mars_ranks[label][0].to_csv(f"ranks_mars_{label}.csv", index=False)
    print(f"\n##### Mars, {label} ({R} m, {len(c)} Dünya penceresi) #####")
    show(mars_ranks[label][0])

### Mars 200 m teşhis görselleri

Haritada keskin bir çizgi ya da tek renkli blok, histogramda ana gruptan kopuk ayrı bir tepe varsa o hedefin 200 m verisinde hata vardır.

In [ ]:
names = [n for n, _, _ in MARS_TARGETS]
fig, ax = plt.subplots(3, len(names), figsize=(4.5 * len(names), 12))
for k, name in enumerate(names):
    z, s = mars_wins[("HRSC_MOLA_200m", name)]
    flag = " (ŞÜPHELİ)" if name in SUSPECT else ""
    ax[0, k].imshow(z, cmap="terrain"); ax[0, k].set_title(name + flag)
    ax[1, k].imshow(s, cmap="magma", vmax=20); ax[1, k].set_title("eğim (derece)")
    ax[2, k].hist(z.ravel(), bins=50); ax[2, k].set_title("yükseklik dağılımı (m)")
plt.tight_layout(); plt.savefig("fig_mars_targets_200m.png", dpi=150); plt.show()

## 6. Doğrulama özeti

- **ilk20_bilinen_%:** En benzer 20 pencerenin yüzde kaçı bilinen analog (yüksek olmalı)
- **AUC:** Rastgele bir bilinen analog penceresinin rastgele bir kontrol penceresinden yüksek skor alma olasılığı (0,5 = yazı-tura, 1,0 = kusursuz)
- **sadece arazi** satırları, çoraklık katmanı olmadan arazinin tek başına ne kadar iş gördüğünü gösterir

In [ ]:
parts = [validate(ranks_moon).assign(veri=f"Ay LOLA {R_MOON} m", katmanlar="arazi + çoraklık"),
         validate(ranks_moon_terrain).assign(veri=f"Ay LOLA {R_MOON} m", katmanlar="sadece arazi")]
for label, (full, terrain) in mars_ranks.items():
    parts.append(validate(full).assign(veri=f"Mars {label}", katmanlar="arazi + çoraklık"))
    parts.append(validate(terrain).assign(veri=f"Mars {label}", katmanlar="sadece arazi"))
val = pd.concat(parts, ignore_index=True)[["veri", "katmanlar", "target", "ilk20_bilinen_%", "AUC"]]
val.to_csv("validation.csv", index=False)
print(val.to_string(index=False))

print("\nÖzet (hedeflerin ortalaması):")
print(val.groupby(["veri", "katmanlar"])[["ilk20_bilinen_%", "AUC"]].mean().round(2))

## 7. İkiz takımı: bir hedefi Dünya'da test etmek için hangi alanlar birlikte gerekir?

Dünya'da hiçbir yer bir Ay ya da Mars hedefine her açıdan benzemez. Bu bölüm, hedefin bütün özelliklerini **birlikte** karşılayan **en az sayıda Dünya alanını** seçer (her alandan en fazla bir pencere; yalnızca en az %50'si çorak, yani gerçekten test yapılabilecek pencereler). Bir özellik, ölçeklenmiş farkı 0,5'ten küçükse (yaklaşık yarım çeyrekler-arası açıklık) "karşılanmış" sayılır.

**Kapsama yüzdesi:** hedefin özelliklerinden kaçının bu takımla karşılandığı (ör. 7 özellikten 4'ü = %57). **Dünya'da karşılanamayan** özellikler de bir bulgudur: o özellik sahada değil, ancak laboratuvarda ya da simülasyonda test edilebilir.

In [ ]:
LABEL_TR = {"relief_m": "rölyef", "slope_mean": "ortalama eğim", "slope_p90": "dik yamaç eğimi (p90)",
            "steep_frac": "15°+ dik alan oranı", "roughness_m": "pürüzlülük", "hypsometry": "hipsometri",
            "w_barren_frac": "çoraklık"}

def twin_team(cand, t, w=W, tol=0.5, max_sites=4, min_barren=0.5):
    # min_barren: takıma yalnızca en az %50'si çorak pencereler girebilir (orman ya da tarla bir test alanı olamaz)
    FS = list(w.index)
    T = t[FS].to_frame().T.astype(float)
    ref = cand[cand["group"].isin(REF_GROUPS)]
    sc = RobustScaler().fit(pd.concat([ref[FS], T]))
    D = np.abs(sc.transform(cand[FS]) - sc.transform(T))   # her pencerenin her özellikte hedefe uzaklığı
    close = D < tol
    covered, team, used = np.zeros(len(FS), bool), [], set()
    while not covered.all() and len(team) < max_sites:
        gain = (close & ~covered).sum(axis=1).astype(float)
        gain[cand["site"].isin(used).values] = -1           # her alandan en fazla bir pencere
        gain[(cand["w_barren_frac"] < min_barren).values] = -1   # bitki örtülü pencereler test alanı olamaz
        best = int(np.argmax(gain - D.mean(axis=1) * 1e-3))  # eşitlikte genel olarak daha yakın olan
        if gain[best] <= 0:
            break
        new = [FS[k] for k in np.where(close[best] & ~covered)[0]]
        r = cand.iloc[best]
        team.append({"target": t["site"], "sira": len(team) + 1, "site": r["site"], "group": r["group"],
                     "lat": round(r["lat"], 3), "lon": round(r["lon"], 3),
                     "karsiladigi": ", ".join(LABEL_TR[f] for f in new)})
        covered |= close[best]
        used.add(r["site"])
    missing = [LABEL_TR[FS[k]] for k in np.where(~covered)[0]]
    coverage = 100 * covered.mean()                         # hedefin özelliklerinin yüzde kaçı Dünya'da karşılanıyor
    return team, missing, coverage

tt_rows, tt_missing = [], []
mars_best = mars_all[(mars_all["source"] == "HRSC_MOLA_200m") & ~mars_all["suspect"]]
e200 = earth_at(int(round(mars_best["res_m"].iloc[0])), earth)
jobs = [(t, cand_moon) for _, t in moon_t.iterrows()] + \
       [(t, e200[~e200["dem_flat"]]) for _, t in mars_best.iterrows()]
for t, c in jobs:
    team, miss, cov = twin_team(c, t)
    tt_rows += team
    tt_missing.append({"target": t["site"], "kapsama_%": round(cov), "dunyada_karsilanamayan": ", ".join(miss) or "-"})
    print(f"\n=== {t['site']}: {len(team)} alanlık ikiz takımı, karşılanan özellik oranı %{cov:.0f} ===")
    for m in team:
        print(f"  {m['sira']}. {m['site']} ({m['lat']}, {m['lon']}) → {m['karsiladigi']}")
    print("  Dünya'da karşılanamayan:", ", ".join(miss) if miss else "yok, tamamı karşılanıyor")

twin_teams = pd.DataFrame(tt_rows).merge(pd.DataFrame(tt_missing), on="target")
twin_teams.to_csv("twin_teams.csv", index=False)

## 7b. Türkiye keşif raporu

Her hedef için Türkiye'deki en benzer pencereler (elle seçilen alanlar ve tarama adayları). Bilinen analogların en iyi skoru referans olarak verilir: bir Türkiye penceresinin skoru buna yakınsa, Türkiye'de o hedef için güçlü ve henüz tanımlanmamış bir test alanı var demektir. Koordinatlar Google Haritalar'da uydu görünümüyle kontrol edilebilir.

In [ ]:
rank_sets = [ranks_moon] + [mars_ranks["HRSC_MOLA_200m"][0]]
allr = pd.concat(rank_sets, ignore_index=True)
tr = allr[allr["group"].isin(["turkey", "scan"])]
known_best = allr[allr["group"] == "known"].groupby("target")["score"].max().rename("bilinen_en_iyi")

disc = (tr.sort_values("score", ascending=False).groupby("target").head(5)
          .merge(known_best, on="target"))
disc["bilinen_en_iyiye_oran_%"] = (100 * disc["score"] / disc["bilinen_en_iyi"]).round(0)
disc["harita"] = "https://www.google.com/maps/@" + disc["lat"].round(4).astype(str) + "," + disc["lon"].round(4).astype(str) + ",12000m/data=!3m1!1e3"
disc = disc[["target", "site", "group", "lat", "lon", "score", "bilinen_en_iyi", "bilinen_en_iyiye_oran_%", "harita"]]
disc.to_csv("turkey_discovery.csv", index=False)

for tname, g in disc.groupby("target"):
    print(f"\n=== {tname}: Türkiye'nin en iyi 5 penceresi (bilinen analogların en iyisi: {g['bilinen_en_iyi'].iloc[0]:.1f}) ===")
    print(g[["site", "group", "lat", "lon", "score", "bilinen_en_iyiye_oran_%"]].round(2).to_string(index=False))

print("\nTüm hedeflerde Türkiye'nin en güçlü 10 penceresi:")
best = tr.sort_values("score", ascending=False).drop_duplicates(["site", "lat", "lon"]).head(10)
print(best[["target", "site", "group", "lat", "lon", "score"]].round(2).to_string(index=False))

## 7c. En iyi test ayı

Ay ve Mars, Dünya'nın her yerinden çok daha soğuk; ama kışın Dünya'daki alanlar karla kaplanır ve test yapılamaz. Bu yüzden her alan için NASA POWER'ın aylık iklim ortalamalarına (sıcaklık, yağış, kar derinliği) bakılır:

1. **Test edilebilir ay:** kar yok (ortalama kar derinliği 1 cm'den az) ve yağış düşük (günde 1 mm'den az). Hiçbir ay bu şartı sağlamıyorsa, karsız aylar arasından en kurak 3 ay alınır. Hiç karsız ay yoksa (NASA POWER'ın ~50 km'lik hücresi buzul ya da kalıcı kar içerebiliyor, ör. Haughton ve Holuhraun) en az karlı aylar kullanılır ve tabloda not düşülür.
2. **En iyi test ayı:** test edilebilir (zaten kurak) aylar arasından hedefe en yakın, yani **en soğuk** ay; sıcaklıklar yakınsa daha kurak olan seçilir.

Bölüm, ikiz takımlarındaki alanlar ve Türkiye'nin en iyi adayları için bir takvim üretir.

In [ ]:
AYLAR = ["Oca", "Şub", "Mar", "Nis", "May", "Haz", "Tem", "Ağu", "Eyl", "Eki", "Kas", "Ara"]

def power_monthly(lat, lon):
    base = ("https://power.larc.nasa.gov/api/temporal/climatology/point?community=RE"
            f"&longitude={lon}&latitude={lat}&format=JSON&parameters=")
    for params in ["T2M,PRECTOTCORR,SNODP", "T2M,PRECTOTCORR"]:     # kar derinliği yoksa onsuz dene
        try:
            p = requests.get(base + params, timeout=60).json()["properties"]["parameter"]
            out = {}
            for m in MONTHS:
                out[f"T_{m}"] = p["T2M"][m]
                out[f"P_{m}"] = p["PRECTOTCORR"][m]
                out[f"S_{m}"] = p["SNODP"][m] if "SNODP" in p else np.nan
            return {k: (np.nan if (v is not None and v < -900) else v) for k, v in out.items()}
        except Exception:
            continue
    raise RuntimeError("NASA POWER verisi alınamadı")

def calendar(row):
    T = np.array([row[f"T_{m}"] for m in MONTHS], float)
    P = np.array([row[f"P_{m}"] for m in MONTHS], float)            # mm/gün
    S = np.array([row[f"S_{m}"] for m in MONTHS], float)            # cm
    snowfree = (S < 1) if not np.isnan(S).all() else (T >= 2)       # kar verisi yoksa: ortalama sıcaklık ≥ 2 °C
    snowfree = np.where(np.isnan(S), T >= 2, snowfree)
    relaxed = False
    if not snowfree.any() and not np.isnan(S).all():
        # POWER'ın ~50 km'lik hücresi buzul ya da kalıcı kar içerebilir (ör. Haughton, Holuhraun):
        # o zaman en az karlı aylar (en düşük kar derinliğinin en fazla 2 katı) kullanılır
        snowfree = S <= max(1.0, 2 * np.nanmin(S))
        relaxed = True
    ok = snowfree & (P < 1.0)
    if not ok.any() and snowfree.any():                              # hiç kurak ay yoksa: karsız aylardan en kurak 3'ü
        idx = np.where(snowfree)[0]
        ok = np.zeros(12, bool); ok[idx[np.argsort(P[idx])[:3]]] = True
    status = np.where(~snowfree, 0, np.where(ok, 2, 1))             # 0 karlı, 1 yağışlı, 2 uygun
    best = None
    if ok.any():
        # uygun aylar zaten kurak; aralarından öncelik en soğuk olana, kuraklık eşitlik bozucu
        score = (T - T.min()) / (np.ptp(T) + 1e-9) + 0.2 * P / (P.max() + 1e-9)
        score[~ok] = np.inf
        best = int(np.argmin(score))
        status[best] = 3                                             # 3 en iyi
    return status, best, T, P, S, relaxed

# aylık iklim (önbellekli)
cm_path = "climate_monthly.csv"
clim_m = pd.read_csv(cm_path) if (os.path.exists(cm_path) and not FORCE) else pd.DataFrame(columns=["site"])
todo = [s for s in SITES if s[0] not in set(clim_m["site"])]
if todo:
    add = []
    for name, lat, lon, group in todo:
        try:
            add.append({"site": name, **power_monthly(lat, lon)})
        except Exception as err:
            print(f"✗ {name}: {err}")
        time.sleep(0.5)
    clim_m = pd.concat([clim_m, pd.DataFrame(add)], ignore_index=True)
    clim_m.to_csv(cm_path, index=False)
print("Kar derinliği verisi:", "var" if clim_m.filter(like="S_").notna().any().any() else "yok (sıcaklıkla tahmin ediliyor)")

# takvim: ikiz takımlarındaki alanlar + Türkiye'nin en iyi adayları
focus = list(dict.fromkeys(list(twin_teams["site"]) + list(disc.drop_duplicates("target")["site"])))
groups = {s[0]: s[3] for s in SITES}
rows, grid_status = [], []
for site in focus:
    r = clim_m[clim_m["site"] == site]
    if r.empty:
        continue
    status, best, T, P, S, relaxed = calendar(r.iloc[0])
    grid_status.append(status)
    uygun = [AYLAR[i] for i in range(12) if status[i] >= 2]
    rows.append({"site": site, "group": groups.get(site, ""),
                 "en_iyi_ay": AYLAR[best] if best is not None else "-",
                 "uygun_aylar": ", ".join(uygun) if uygun else "-",
                 "en_iyi_ay_sicaklik_C": round(T[best], 1) if best is not None else np.nan,
                 "en_iyi_ay_yagis_mm_gun": round(P[best], 2) if best is not None else np.nan,
                 "not": "hiç karsız ay yok; en az karlı aylar kullanıldı" if relaxed else ""})
best_months = pd.DataFrame(rows)
best_months.to_csv("best_months.csv", index=False)
print(best_months.to_string(index=False))

# takvim görseli
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(11, 0.45 * len(rows) + 1.5))
cmap = ListedColormap(["#a9cce3", "#d5d8dc", "#a9dfbf", "#1e8449"])
ax.imshow(np.array(grid_status), cmap=cmap, vmin=0, vmax=3, aspect="auto")
ax.set_xticks(range(12)); ax.set_xticklabels(AYLAR)
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r["site"] for r in rows])
for i, r in enumerate(rows):
    for j in range(12):
        if grid_status[i][j] == 3:
            ax.text(j, i, "★", ha="center", va="center", color="white", fontsize=12)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(cmap.colors, ["karlı", "yağışlı", "uygun", "en iyi"])],
          loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False)
ax.set_title("En iyi test ayı: karsız, kurak ve en soğuk ay (★)")
plt.tight_layout(); plt.savefig("fig_best_month.png", dpi=150); plt.show()

## 7d. Karşılaştırma kartları: hedef, Türkiye adayı ve gerçek uydu görüntüsü

Her hedef için Türkiye'nin en benzer penceresi dört panelle gösterilir (hepsi aynı 12 km ölçekte):

1. **Hedef:** Ay ya da Mars bölgesinin gölgelendirilmiş arazisi (NASA LOLA / MOLA + HRSC)
2. **Türkiye adayı:** aynı ölçekte gölgelendirilmiş arazi (NASA NASADEM / Copernicus)
3. **Gerçek görüntü:** Sentinel-2 gerçek renkli uydu görüntüsü (ESA; AWS'teki açık arşivden, 2025 yazının en az bulutlu sahnesi)
4. **Çoraklık:** ESA WorldCover arazi örtüsü (kahverengi = çorak, yeşil = bitki örtülü, mavi = su)

Panel 1 ve 2 modelin "gördüğü" şeydir; panel 3 ve 4 gerçekte orada ne olduğunu gösterir.

In [ ]:
!pip install -q pystac-client
from pystac_client import Client
from rasterio.enums import Resampling as RS
from matplotlib.colors import ListedColormap

def hillshade(z, res, az=315, alt=45):
    gy, gx = np.gradient(z, res)
    slope = np.pi / 2 - np.arctan(np.hypot(gx, gy))
    aspect = np.arctan2(-gx, gy)
    a, b = np.radians(az), np.radians(alt)
    return np.clip(np.sin(b) * np.sin(slope) + np.cos(b) * np.cos(slope) * np.cos(a - aspect), 0, 1)

def to_crs_bounds(dst_crs, lat, lon, half_m):
    tf = Transformer.from_crs("EPSG:4326", CRS.from_wkt(dst_crs.to_wkt()), always_xy=True)
    x, y = tf.transform(lon, lat)
    return x - half_m, y - half_m, x + half_m, y + half_m

def earth_dem_window(site, dem, lat, lon, half_m=WIN_M / 2):
    with rasterio.open(f"dem/{site}_{dem}_30m.tif") as src:
        win = from_bounds(*to_crs_bounds(src.crs, lat, lon, half_m), src.transform)
        z = src.read(1, window=win).astype(float)
    z = np.where(np.isnan(z), np.nanmean(z), z)
    return z, 30.0

S2 = Client.open("https://earth-search.aws.element84.com/v1")

def s2_truecolor(lat, lon, half_m=WIN_M / 2, dates="2025-06-01/2025-09-30", px=600):
    dlat = half_m / 111000
    dlon = half_m / (111000 * np.cos(np.radians(lat)))
    bbox = [lon - dlon, lat - dlat, lon + dlon, lat + dlat]
    try:
        items = list(S2.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=dates,
                               query={"eo:cloud_cover": {"lt": 10}}, max_items=30).items())
    except Exception:                                           # sunucu filtreyi desteklemezse filtresiz ara
        items = [it for it in S2.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=dates,
                                        max_items=60).items() if it.properties.get("eo:cloud_cover", 100) < 10]
    items.sort(key=lambda it: it.properties.get("eo:cloud_cover", 100))
    for it in items:
        try:
            with rasterio.open(it.assets["visual"].href) as src:
                win = from_bounds(*to_crs_bounds(src.crs, lat, lon, half_m), src.transform)
                img = src.read([1, 2, 3], window=win, out_shape=(3, px, px), resampling=RS.bilinear,
                               boundless=True, fill_value=0)
            if (img.sum(axis=0) == 0).mean() < 0.02:            # sahne pencereyi tamamen kapsamalı
                return np.moveaxis(img, 0, -1), it.datetime.date(), it.properties.get("eo:cloud_cover")
        except Exception:
            continue
    return None, None, None

def worldcover_rgb(lat, lon, half_km=WIN_M / 2000, px=600):
    dlat = half_km / 111.0
    dlon = half_km / (111.0 * np.cos(np.radians(lat)))
    with rasterio.open(wc_tile(lat, lon)) as src:
        win = from_bounds(lon - dlon, lat - dlat, lon + dlon, lat + dlat, src.transform)
        lc = src.read(1, window=win, out_shape=(px, px), resampling=Resampling.nearest, boundless=True, fill_value=0)
    cls = np.full(lc.shape, 3)                                  # 3 = diğer
    cls[np.isin(lc, [60, 70, 100])] = 0                         # çorak
    cls[np.isin(lc, [10, 20, 30, 40, 90, 95])] = 1              # bitki örtüsü
    cls[lc == 80] = 2                                           # su
    return cls

TARGET_WINS = {**moon_wins, **{n: v for (lab, n), v in mars_wins.items() if lab == "HRSC_MOLA_200m"}}
TARGET_RES = {**dict(zip(moon_t["site"], moon_t["res_m"])),
              **dict(zip(mars_best["site"], mars_best["res_m"]))}
dem_of = earth.drop_duplicates("site").set_index("site")["dem"].to_dict()

picks = disc.sort_values("score", ascending=False).drop_duplicates("target")
cards = []
for _, p in picks.iterrows():
    print(f"{p['target']} ↔ {p['site']} ({p['lat']:.3f}, {p['lon']:.3f})")
    card = {"target": p["target"], "site": p["site"], "lat": p["lat"], "lon": p["lon"], "score": p["score"]}
    res_t = TARGET_RES.get(p["target"], 30.0)
    ex = 5.0 if p["target"].startswith("Mars") else 1.0        # Mars bölgeleri çok düz: iki panele de aynı düşey abartı
    card["ex"], card["res_t"] = ex, res_t
    try:
        zt, st = TARGET_WINS[p["target"]]
        card["t_hs"] = hillshade(zt * ex, res_t)
    except Exception as err:
        print("   hedef:", err)
    try:
        ze, re_ = earth_dem_window(p["site"], dem_of[p["site"]], p["lat"], p["lon"])
        ze = ndimage.zoom(ze, re_ / res_t, order=1, mode="nearest")   # modelin gördüğü gibi: hedefin çözünürlüğünde
        card["e_hs"] = hillshade(ze * ex, res_t)
    except Exception as err:
        print("   aday arazisi:", err)
    try:
        card["rgb"], card["rgb_date"], card["rgb_cloud"] = s2_truecolor(p["lat"], p["lon"])
        print("   Sentinel-2:", card["rgb_date"], f"bulut %{card['rgb_cloud']}" if card["rgb_date"] else "bulunamadı")
    except Exception as err:
        print("   Sentinel-2:", err)
    try:
        card["wc"] = worldcover_rgb(p["lat"], p["lon"])
    except Exception as err:
        print("   WorldCover:", err)
    cards.append(card)

wc_cmap = ListedColormap(["#c8a165", "#3a7d44", "#4a90d9", "#d9d9d9"])
fig, axes = plt.subplots(len(cards), 4, figsize=(17, 4.3 * len(cards)))
axes = np.atleast_2d(axes)
for i, c in enumerate(cards):
    ex_txt = f", düşey abartı {c.get('ex', 1):.0f}×" if c.get("ex", 1) != 1 else ""
    titles = [f"Hedef: {c['target']}\n{c.get('res_t', 0):.0f} m{ex_txt}",
              f"Türkiye adayı: {c['site']}\nskor {c['score']:.1f} · aynı çözünürlük{ex_txt}",
              f"Sentinel-2 gerçek renk ({c.get('rgb_date') or '-'})", "WorldCover: çorak / bitki / su"]
    panels = [("t_hs", "gray"), ("e_hs", "gray"), ("rgb", None), ("wc", wc_cmap)]
    for j, (key, cmap) in enumerate(panels):
        ax = axes[i, j]
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(titles[j], fontsize=10)
        v = c.get(key)
        if v is None:
            ax.text(0.5, 0.5, "görüntü yok", ha="center", va="center", transform=ax.transAxes)
            continue
        if key == "wc":
            ax.imshow(v, cmap=cmap, vmin=0, vmax=3)
        elif key == "rgb":
            ax.imshow(v)
        else:
            ax.imshow(v, cmap=cmap, vmin=0, vmax=1)
plt.tight_layout(rect=(0, 0.02, 1, 1))
fig.text(0.5, 0.005, "Her panel 12 km × 12 km. Görüntü: Sentinel-2 L2A (ESA/Copernicus, AWS Open Data). "
         "Arazi: NASA LRO LOLA, MGS MOLA + ESA HRSC, NASA NASADEM / Copernicus DEM. Arazi örtüsü: ESA WorldCover 2021.",
         ha="center", fontsize=9)
plt.savefig("fig_candidate_cards.png", dpi=150); plt.show()

pd.DataFrame([{k: c.get(k) for k in ["target", "site", "lat", "lon", "score", "rgb_date", "rgb_cloud"]} for c in cards]) \
  .to_csv("candidate_cards.csv", index=False)

## 7e. Kayaç katmanı (sondaj modu için)

Her pencereye ve hedefe bir **kayaç ailesi** atanır. Karşılaştırma aile düzeyindedir, mineral düzeyinde değildir.

| Aile | Dünya (Macrostrat kayaç adı) | Mars (USGS SIM 3292 birimi) | Ay (USGS birleşik harita) |
| --- | --- | --- | --- |
| volkanik | bazalt, volkanik, cüruf, lav… | "volcanic" birimleri | mare birimleri |
| tortul | kumtaşı, kireçtaşı, alüvyon, playa… | "transition" birimleri | - |
| çarpma | çarpma breşi, krater dolgusu | "impact" birimleri | krater, havza ve düzlük birimleri |
| kristalen | şist, gnays, mermer, derinlik kayaçları… | - | yüksek arazi (terra) |
| karışık yüksek arazi | - | "highland" birimleri (çarpma + volkanik + tortul karışımı) | - |

- **Dünya:** Macrostrat her nokta için birden fazla haritadan cevap verir; en ayrıntılı haritadan başlanır, kayaç adı boşsa ya da sınıflanamıyorsa bir sonrakine geçilir. Her 30 m penceresinin merkezi bir kez sorgulanır ve `rock_earth_v2.csv` dosyasına kaydedilir (ilk çalıştırmada birkaç dakika sürer).
  - **Temel kayacı haritası kullanılmaz:** Macrostrat'ın 463 numaralı kaynağı yüzeyi değil yerin altındaki temel kayacını gösteriyor (ör. Kansas'ı granitoid gösteriyordu); bu kaynak dışarıda bırakılır.
  - **Çarpma önceliği:** Çarpma yapıları yalnızca bazı haritalarda çizilir. Haritalardan herhangi biri noktayı çarpma yapısı olarak gösteriyorsa aile "çarpma" kabul edilir.
- **Mars:** Hedef noktasının içinde kaldığı birim (USGS Astrogeology API).
  - **Gale düzeltmesi:** 1:20 milyon ölçekli haritada Gale kraterinin tamamı "çarpma birimi" olarak çizilmiş; Curiosity'nin incelediği katmanlar ise tortul (kil ve sülfat içeren katmanlar; JPL, 17 Ağustos 2021). Gale'in ailesi görev bulgusuna göre "tortul" alınır; harita sonucu da tabloda ayrıca gösterilir.
- **Ay:** USGS birleşik Ay jeoloji haritası bir kez indirilir (224 MB, `geo/` klasörüne). Hedef noktasının içinde kaldığı birim seçilir.
- Sınıflanamayan pencereler "bilinmiyor" olur ve hiçbir hedefle eşleşmez.

**Sınır:** Mars haritası 1:20 milyon ölçekli ve kayaç değil yaş-köken birimi veriyor; Türkiye için Macrostrat'ın kullandığı haritalar kaba ölçekli.

In [ ]:
import requests, zipfile, glob, re
from concurrent.futures import ThreadPoolExecutor
import geopandas as gpd
from shapely.geometry import Point, shape

FAM_TR = {"volcanic": "volkanik", "sedimentary": "tortul", "impact": "çarpma",
          "crystalline": "kristalen", "mixed_highland": "karışık yüksek arazi", None: "bilinmiyor"}
KW = {"impact": ["impact", "crater fill"],
      "volcanic": ["basalt", "volcanic", "scoria", "lava", "tholeiite", "basanite", "andesite", "rhyolite",
                   "dacite", "tuff", "ignimbrite", "pyroclastic", "trachy", "cinder"],
      "sedimentary": ["sediment", "sandstone", "limestone", "dolomite", "shale", "alluvi", "playa", "lake",
                      "lacustrine", "evaporite", "gypsum", "halite", "rock salt", "conglomerate", "flysch", "unconsolidated",
                      "marl", "clastic", "carbonate", "mudstone", "siltstone", "clay", "gravel", "terrace",
                      "eolian", "aeolian", "fluvial", "colluvium", "till", "glacial", "moraine", "travertine", "chalk"],
      "crystalline": ["metamorphic", "schist", "gneiss", "marble", "greenschist", "amphibolite", "intrusive",
                      "granit", "plutonic", "diorite", "gabbro", "quartzite", "migmatite", "crystalline",
                      "serpentin", "peridotite", "ultramafic", "slate", "phyllite"]}
ORDER = ["impact", "volcanic", "sedimentary", "crystalline"]

def classify_earth(text):
    # en çok anahtar kelime eşleşen aile; eşitlikte ORDER sırası
    t = str(text).lower()
    hits = {f: sum(k in t for k in KW[f]) for f in ORDER}
    best = max(ORDER, key=lambda f: (hits[f], -ORDER.index(f)))
    return best if hits[best] > 0 else None

EXCLUDE_SOURCES = {463}   # yüzey değil, temel kayacı haritası

def macrostrat_point(lat, lon):
    units = []
    for attempt in range(3):
        try:
            r = requests.get("https://macrostrat.org/api/v2/geologic_units/map",
                             params={"lat": round(lat, 4), "lng": round(lon, 4)}, timeout=30)
            units = r.json().get("success", {}).get("data", [])
            break
        except Exception:
            units = []
    units = [u for u in units if u.get("source_id") not in EXCLUDE_SOURCES]
    fams = [classify_earth(u.get("lith")) or classify_earth(u.get("name")) for u in units]
    if "impact" in fams:                 # çarpma önceliği: herhangi bir harita çarpma yapısı gösteriyorsa
        u = units[fams.index("impact")]
        return {"rock_family": "impact", "rock_source": u.get("source_id"),
                "rock_name": str(u.get("name"))[:80], "rock_lith": str(u.get("lith"))[:120]}
    for u, fam in zip(units, fams):      # en ayrıntılı haritadan başla
        if fam:
            return {"rock_family": fam, "rock_source": u.get("source_id"),
                    "rock_name": str(u.get("name"))[:80], "rock_lith": str(u.get("lith"))[:120]}
    return {"rock_family": None, "rock_source": None, "rock_name": None, "rock_lith": None}

# ---------- Dünya: her 30 m penceresi ----------
path = "rock_earth_v2.csv"   # v2: temel kayacı haritası çıkarıldı, çarpma önceliği eklendi
rock_earth = pd.read_csv(path) if (os.path.exists(path) and not FORCE) else pd.DataFrame(columns=["site", "lat", "lon"])
done = set(zip(rock_earth["site"], rock_earth["lat"].round(5), rock_earth["lon"].round(5)))
todo = earth[[(s, round(a, 5), round(b, 5)) not in done for s, a, b in zip(earth["site"], earth["lat"], earth["lon"])]]
if len(todo):
    print(f"Macrostrat sorgulanıyor: {len(todo)} pencere...")
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(lambda p: macrostrat_point(*p), zip(todo["lat"], todo["lon"])))
    new = pd.concat([todo[["site", "group", "lat", "lon"]].reset_index(drop=True), pd.DataFrame(res)], axis=1)
    rock_earth = pd.concat([rock_earth, new], ignore_index=True)
    rock_earth.to_csv(path, index=False)
rock_earth = rock_earth[rock_earth["site"].isin(set(earth["site"]))]
rock_earth["rock_family"] = rock_earth["rock_family"].where(rock_earth["rock_family"].notna(), None)

def add_rock(df):
    # her pencereye aynı alandaki en yakın 30 m penceresinin kayaç ailesini taşır
    out = df.drop(columns=["rock_family"], errors="ignore")
    return out.join(nearest_join(out, rock_earth, ["rock_family"]))

print("\nAlanlara göre kayaç aileleri (pencere sayısı):")
summ_rock = (rock_earth.assign(aile=rock_earth["rock_family"].map(lambda f: FAM_TR.get(f, "bilinmiyor")))
             .groupby(["group", "site"])["aile"].agg(lambda s: ", ".join(f"{k} {v}" for k, v in s.value_counts().items())))
print(summ_rock[summ_rock.index.get_level_values(0) != "scan"].to_string())
print("\nKontrol ve bilinen alanlarda Macrostrat'ın döndürdüğü kayaç adları (kontrol için):")
chk = rock_earth[rock_earth["group"].isin(["control", "known"])]
print(chk.groupby(["site", "rock_name", "rock_lith"], dropna=False).size().rename("pencere").reset_index()
      .to_string(index=False, max_colwidth=60))
print("\nTarama adayları, ailelere göre pencere sayısı:")
print(rock_earth[rock_earth["group"] == "scan"]["rock_family"].map(lambda f: FAM_TR.get(f, "bilinmiyor")).value_counts().to_string())

# ---------- Mars: hedef noktasının içinde kaldığı birim ----------
def classify_mars(desc):
    d = str(desc).lower()
    for key, fam in [("impact", "impact"), ("volcanic", "volcanic"), ("transition", "sedimentary"), ("highland", "mixed_highland")]:
        if key in d:
            return fam
    return None

MARS_GEO = "https://astrogeology.usgs.gov/pygeoapi/collections/mars/sim3292_global_geologic_map/units/items"
rock_t = []
for name, lat, lon in MARS_TARGETS:
    r = requests.get(MARS_GEO, params={"bbox": f"{lon-0.01},{lat-0.01},{lon+0.01},{lat+0.01}", "f": "json", "limit": 20}, timeout=60)
    feats = r.json().get("features", [])
    inside = [f for f in feats if f.get("geometry") and shape(f["geometry"]).contains(Point(lon, lat))]
    exact = len(inside) > 0
    p = (inside or feats or [{"properties": {}}])[0]["properties"]
    rock_t.append({"site": name, "body": "Mars", "unit": p.get("unit"), "unit_desc": p.get("unitdesc"),
                   "rock_family": classify_mars(p.get("unitdesc")), "tam_icinde": exact})

# ---------- Ay: USGS birleşik jeoloji haritası ----------
def classify_moon(text):
    t = str(text).lower()
    if "mare" in t:
        return "volcanic"
    if any(k in t for k in ["crater", "basin", "plains"]):
        return "impact"
    if any(k in t for k in ["terra", "massif", "highland"]):
        return "crystalline"
    return None

os.makedirs("geo", exist_ok=True)
mz = "geo/Unified_Geologic_Map_of_the_Moon_GIS_v2.zip"
if not os.path.exists(mz):
    print("Ay jeoloji haritası indiriliyor (224 MB)...")
    with requests.get("https://asc-astropedia.s3.us-west-2.amazonaws.com/Moon/Geology/Unified_Geologic_Map_of_the_Moon_GIS_v2.zip",
                      stream=True, timeout=600) as r:
        with open(mz, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
if not os.path.isdir("/content/moon_geo"):
    zipfile.ZipFile(mz).extractall("/content/moon_geo")
try:
    import pyogrio
    list_layers = lambda src: list(pyogrio.list_layers(src)[:, 0])
except ImportError:
    import fiona
    list_layers = fiona.listlayers

# Zip'teki bütün vektör dosyalarını dene; açılamayanları atla (ör. sıkıştırılmış geodatabase)
srcs = sorted(set(glob.glob("/content/moon_geo/**/*.shp", recursive=True) + glob.glob("/content/moon_geo/**/*.gpkg", recursive=True)
                  + glob.glob("/content/moon_geo/**/*.gdb", recursive=True)))
layers, failed = [], []
for src in srcs:
    for path in ([src, "OpenFileGDB:" + src] if src.endswith(".gdb") else [src]):
        try:
            layers += [(path, l) for l in list_layers(path)]
            break
        except Exception as err:
            failed.append(f"{os.path.relpath(src, '/content/moon_geo')}: {str(err)[:80]}")
if failed:
    print("Açılamayan dosyalar:", *failed, sep="\n  ")
if not layers:
    print("Zip içeriği (ilk 40 dosya):")
    for f in sorted(glob.glob("/content/moon_geo/**/*", recursive=True))[:40]:
        print("  ", os.path.relpath(f, "/content/moon_geo"))
    raise RuntimeError("Ay haritasında okunabilen vektör dosyası bulunamadı; yukarıdaki listeyi gönder.")
print("Okunabilen katmanlar:", [os.path.basename(p) + " → " + l for p, l in layers][:20])
unit_layers = [x for x in layers if re.search("unit", x[1], re.I) and not re.search("contact|line|label|anno", x[1], re.I)] \
              or [x for x in layers if re.search("unit|geo", x[1] + x[0], re.I)] or layers
print("Kullanılan katman:", unit_layers[0][1])
moon_geo = gpd.read_file(unit_layers[0][0], layer=unit_layers[0][1])
print("Ay birim tablosu sütunları:", list(moon_geo.columns))
from shapely.ops import transform as shp_transform
from pyproj import Transformer as _Tr
def south_stereo(x, y):
    # güney kutbu stereografik izdüşüm (Ay yarıçapı 1737,4 km); kutup çevresindeki birimleri doğru karşılaştırmak için
    lon, lat = np.radians(np.asarray(x, float)), np.radians(np.asarray(y, float))
    rho = 2 * 1737400.0 * np.tan(np.pi / 4 + lat / 2)
    return rho * np.sin(lon), rho * np.cos(lon)
if moon_geo.crs is None or moon_geo.crs.is_geographic:
    mg = moon_geo.cx[:, -90:-60].copy()
    mg["geometry"] = mg.geometry.apply(lambda g: shp_transform(south_stereo, g).buffer(0))
    to_pt = lambda lon, lat: Point(*south_stereo(lon, lat))
else:
    mg = moon_geo
    _t = _Tr.from_crs(moon_geo.crs.geodetic_crs, moon_geo.crs, always_xy=True)
    to_pt = lambda lon, lat: Point(*_t.transform(lon, lat))
print("Ay haritası CRS:", moon_geo.crs.name if moon_geo.crs is not None else "yok", "| güney kutbu birim sayısı:", len(mg))
txt_cols = [c for c in moon_geo.columns if moon_geo[c].dtype == object and c != "geometry"]
for name, lat, lon in MOON_TARGETS:
    pt = to_pt(lon, lat)
    hit = mg[mg.contains(pt)]
    exact = len(hit) > 0
    if not exact:
        hit = mg.iloc[[int(mg.distance(pt).values.argmin())]]
    row = hit.iloc[0]
    text = " | ".join(str(row[c]) for c in txt_cols)
    rock_t.append({"site": name, "body": "Moon", "unit": text[:60], "unit_desc": text[:200],
                   "rock_family": classify_moon(text), "tam_icinde": exact})

rock_targets = pd.DataFrame(rock_t)
# Harita ölçeği yetersiz kaldığında görev bulgusuyla düzeltme (kaynağıyla)
ROCK_OVERRIDE = {"Mars_Gale_Curiosity": ("sedimentary", "Harita: çarpma birimi (1:20 milyon ölçekte kraterin tamamı). "
                 "Curiosity'nin incelediği katmanlar tortul: kil ve sülfat içeren katmanlar (JPL, 17 Ağustos 2021).")}
rock_targets["harita_aile"] = rock_targets["rock_family"].map(lambda f: FAM_TR.get(f, "bilinmiyor"))
rock_targets["duzeltme"] = ""
for site, (fam, note) in ROCK_OVERRIDE.items():
    m = rock_targets["site"] == site
    rock_targets.loc[m, "rock_family"] = fam
    rock_targets.loc[m, "duzeltme"] = note
rock_targets["aile"] = rock_targets["rock_family"].map(lambda f: FAM_TR.get(f, "bilinmiyor"))
rock_targets.to_csv("rock_targets.csv", index=False)
print("\nHedeflerin kayaç aileleri (tam_icinde=False ise en yakın birim alındı):")
print(rock_targets[["site", "unit", "harita_aile", "aile", "tam_icinde"]].to_string(index=False))
for _, r in rock_targets[rock_targets["duzeltme"] != ""].iterrows():
    print(f"  Düzeltme, {r['site']}: {r['duzeltme']}")

fam_map = dict(zip(rock_targets["site"], rock_targets["rock_family"]))
moon_t["rock_family"] = moon_t["site"].map(fam_map)
mars_all["rock_family"] = mars_all["site"].map(fam_map)
cand_moon = add_rock(cand_moon)
LABEL_TR["rock_match"] = "kayaç ailesi"

def with_rock(c, t):
    # kayaç eşleşmesi: hedefle aynı ailedeyse 1, değilse 0 (hedefin kendisi 1)
    c2 = c.assign(rock_match=((c["rock_family"] == t["rock_family"]) & c["rock_family"].notna()).astype(float).values)
    t2 = t.copy(); t2["rock_match"] = 1.0
    return c2, t2

def rank_any(c, tg, w):
    if "rock_match" not in w.index:
        return rank(c, tg, w)
    out = []
    for _, t in tg.iterrows():
        c2, t2 = with_rock(c, t)
        out.append(rank(c2, pd.DataFrame([t2]), w))
    return pd.concat(out, ignore_index=True)

print("\nHer hedefle aynı kayaç ailesindeki alanlar:")
for _, t in rock_targets.iterrows():
    same = rock_earth[(rock_earth["rock_family"] == t["rock_family"]) & rock_earth["rock_family"].notna()]
    print(f"  {t['site']} ({t['aile']}): {same['site'].nunique()} alan, {len(same)} pencere; "
          f"Türkiye: {', '.join(sorted(set(same[same['group'].isin(['turkey'])]['site'])))[:150] or '-'}; "
          f"tarama penceresi: {(same['group'] == 'scan').sum()}")

## 7f. Test amacı modları: aynı hedef, farklı testler

Aynı hedef için yapılacak teste göre bazı arazi özelliklerinin farkı **2 kat** sayılır. Çoraklığın toplam ağırlıktaki payı her modda genel moddakiyle aynı tutulur (genel 2; iniş ve rover 3,09; üs 2,91; sondaj 2,73). Böylece modlar yalnızca arazi özelliklerinin kendi arasındaki dengeyi değiştirir.

| Mod | Ne için | 2 kat sayılan özellikler |
| --- | --- | --- |
| Genel | Genel arazi benzerliği (önceki bölümlerdeki ağırlıklar) | (yok) |
| İniş | İniş sistemleri, iniş alanı seçimi | ortalama eğim, 15°+ dik alan oranı, pürüzlülük |
| Rover | Tekerlek ve hareket testleri | dik yamaç eğimi (p90), 15°+ dik alan oranı, pürüzlülük |
| Üs | Üs kurulumu, büyük ölçekli arazi şekli | rölyef, hipsometri, ortalama eğim |
| Sondaj | Sondaj ve örnekleme | kayaç ailesi eşleşmesi (yeni özellik, 7e) |

Bu bölüm üç şey hesaplar:
1. **Doğrulama:** Her mod için ilk-20 isabeti ve AUC. Bir modun AUC'si genel moddan 0,05'ten fazla düşerse o mod işaretlenir ve ağırlıkları gözden geçirilmelidir.
2. **En iyi pencere:** Her hedef × mod için hem dünyadaki hem Türkiye'deki en benzer pencere. Mod değişince en iyi alan değişiyorsa tabloda "değişti" yazar.
3. **Mod ikiz takımı:** Her mod için, o testin **öncelikli özelliklerini** (2 kat sayılanlar ve çoraklık) birlikte karşılayan en az sayıda alan. Genel modun tam ikiz takımı 7. bölümdedir.

Sondaj modunda arazi özellikleri genel moddaki ağırlıklarla kalır; kayaç ailesi eşleşmesi (aynı aile 1, değil 0) 2 ağırlıkla eklenir. Mod ikiz takımında sondajın öncelikli özellikleri kayaç ailesi ve çoraklıktır.

In [ ]:
BARREN_SHARE = W["w_barren_frac"] / W.sum()   # genel modda çoraklığın toplam ağırlıktaki payı

def mode_w(double):
    # öncelikli arazi özellikleri 2 kat; çoraklığın payı genel moddakiyle aynı tutulur,
    # böylece modlar yalnızca arazi özelliklerinin kendi arasındaki dengeyi değiştirir
    w = W.copy()
    for f in double:
        w[f] = (w[f] if f in w.index else 1.0) * 2   # kayaç eşleşmesi yeni özellik: 1 → 2
    terrain = w.drop("w_barren_frac").sum()
    w["w_barren_frac"] = BARREN_SHARE * terrain / (1 - BARREN_SHARE)
    return w

PRIORITY = {"genel": [],
            "inis":  ["slope_mean", "steep_frac", "roughness_m"],
            "rover": ["slope_p90", "steep_frac", "roughness_m"],
            "us":    ["relief_m", "hypsometry", "slope_mean"],
            "sondaj": ["rock_match"]}
MODES = {m: mode_w(p) for m, p in PRIORITY.items()}
MODE_TR = {"genel": "Genel", "inis": "İniş", "rover": "Rover", "us": "Üs", "sondaj": "Sondaj"}
print("Mod ağırlıkları (çoraklığın payı her modda %{:.1f}):".format(100 * BARREN_SHARE))
print(pd.DataFrame(MODES).T.reindex(columns=list(W.index) + ["rock_match"]).fillna(0).round(2).to_string())

# Her veri seti için hedefler ve aday Dünya pencereleri
mars_sets = {}
for label in MARS_SOURCES:
    tg = mars_all[(mars_all["source"] == label) & ~mars_all["suspect"]]
    e = earth_at(int(round(tg["res_m"].iloc[0])), earth)
    mars_sets[label] = (tg, add_rock(e[~e["dem_flat"]].copy()))
DATASETS = {f"Ay LOLA {R_MOON} m": (moon_t, cand_moon)}
DATASETS.update({f"Mars {label}": v for label, v in mars_sets.items()})
TWIN_SETS = [f"Ay LOLA {R_MOON} m", "Mars HRSC_MOLA_200m"]   # ikiz takımı 7. bölümdeki veri setleriyle aynı

# 1) Doğrulama ve 2) en iyi pencereler
mv, mb, mode_ranks = [], [], {}
for ds, (tg, c) in DATASETS.items():
    for m, w in MODES.items():
        r = rank_any(c, tg, w)
        mode_ranks[(ds, m)] = r
        mv.append(validate(r).assign(veri=ds, mod=MODE_TR[m]))
        for t, g in r.groupby("target"):
            best_all = g.loc[g["score"].idxmax()]
            gtr = g[g["group"].isin(["turkey", "scan"])]
            best_tr = gtr.loc[gtr["score"].idxmax()] if len(gtr) else None
            mb.append({"veri": ds, "target": t, "mod": MODE_TR[m],
                       "en_iyi_alan": best_all["site"], "en_iyi_lat": round(best_all["lat"], 3),
                       "en_iyi_lon": round(best_all["lon"], 3), "en_iyi_skor": round(best_all["score"], 1),
                       "tr_en_iyi_alan": None if best_tr is None else best_tr["site"],
                       "tr_lat": None if best_tr is None else round(best_tr["lat"], 3),
                       "tr_lon": None if best_tr is None else round(best_tr["lon"], 3),
                       "tr_skor": None if best_tr is None else round(best_tr["score"], 1)})

mode_val = pd.concat(mv, ignore_index=True)[["veri", "mod", "target", "ilk20_bilinen_%", "AUC"]]
mode_val.to_csv("modes_validation.csv", index=False)
summ = mode_val.groupby(["veri", "mod"], sort=False)[["ilk20_bilinen_%", "AUC"]].mean().round(2).reset_index()
genel_auc = summ[summ["mod"] == "Genel"].set_index("veri")["AUC"]
summ["AUC_farki_genel"] = (summ["AUC"] - summ["veri"].map(genel_auc)).round(2)
summ["uyari"] = np.where(summ["AUC_farki_genel"] < -0.05, "AUC düştü: ağırlıkları gözden geçir", "")
summ.to_csv("modes_validation_summary.csv", index=False)
print("\n=== Modlara göre doğrulama (hedeflerin ortalaması) ===")
print(summ.to_string(index=False))

mode_best = pd.DataFrame(mb)
gen = mode_best[mode_best["mod"] == "Genel"].set_index(["veri", "target"])
key = list(zip(mode_best["veri"], mode_best["target"]))
mode_best["dunya_pencere_degisti"] = [a != gen.loc[k, "en_iyi_alan"] or (la, lo) != (gen.loc[k, "en_iyi_lat"], gen.loc[k, "en_iyi_lon"])
                              for k, a, la, lo in zip(key, mode_best["en_iyi_alan"], mode_best["en_iyi_lat"], mode_best["en_iyi_lon"])]
mode_best["tr_pencere_degisti"] = [(la, lo) != (gen.loc[k, "tr_lat"], gen.loc[k, "tr_lon"])
                           for k, la, lo in zip(key, mode_best["tr_lat"], mode_best["tr_lon"])]
mode_best.to_csv("modes_best_windows.csv", index=False)
print("\n=== Her hedef × mod için en iyi pencere (dünya ve Türkiye) ===")
for (ds, t), g in mode_best.groupby(["veri", "target"], sort=False):
    print(f"\n--- {t} ({ds}) ---")
    print(g[["mod", "en_iyi_alan", "en_iyi_skor", "dunya_pencere_degisti", "tr_en_iyi_alan", "tr_lat", "tr_lon", "tr_skor", "tr_pencere_degisti"]]
          .to_string(index=False))

# 3) Mod ikiz takımları: yalnızca o modun öncelikli özellikleri (2 kat sayılanlar + çoraklık)
mt_rows = []
for ds in TWIN_SETS:
    tg, c = DATASETS[ds]
    for m, w in MODES.items():
        if m == "genel":
            continue
        wp = w[PRIORITY[m] + ["w_barren_frac"]]
        for _, t in tg.iterrows():
            c_t, t_t = with_rock(c, t) if "rock_match" in wp.index else (c, t)
            team, miss, cov = twin_team(c_t, t_t, w=wp)
            mt_rows.append({"veri": ds, "target": t["site"], "mod": MODE_TR[m],
                            "oncelikli_ozellikler": ", ".join(LABEL_TR[f] for f in wp.index),
                            "takim": " + ".join(f"{x['site']} ({x['lat']}, {x['lon']})" for x in team) or "-",
                            "alan_sayisi": len(team), "kapsama_%": round(cov),
                            "karsilanamayan": ", ".join(miss) or "-"})
mode_twins = pd.DataFrame(mt_rows)
mode_twins.to_csv("modes_twin_teams.csv", index=False)
print("\n=== Mod ikiz takımları (öncelikli özellikler) ===")
for t, g in mode_twins.groupby("target", sort=False):
    print(f"\n--- {t} ---")
    for _, r in g.iterrows():
        print(f"  {r['mod']}: {r['takim']} | kapsama %{r['kapsama_%']} | karşılanamayan: {r['karsilanamayan']}")

# Görsel: modlara göre Türkiye'nin en iyi skoru
pv = mode_best[mode_best["veri"].isin(TWIN_SETS)].pivot_table(index="target", columns="mod", values="tr_skor", sort=False)[[MODE_TR[m] for m in MODES]]
fig, ax = plt.subplots(figsize=(10, 0.5 * len(pv) + 1.5))
im = ax.imshow(pv.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels(pv.columns)
ax.set_yticks(range(pv.shape[0])); ax.set_yticklabels(pv.index)
for i in range(pv.shape[0]):
    for j in range(pv.shape[1]):
        v = pv.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", color="white" if v < pv.values[~np.isnan(pv.values)].mean() else "black")
ax.set_title(f"Türkiye'nin en iyi penceresinin skoru, test amacına göre (Ay {R_MOON} m, Mars 200 m)")
plt.colorbar(im, ax=ax, label="benzerlik skoru (0-100)")
plt.tight_layout(); plt.savefig("fig_modes.png", dpi=150); plt.show()

## 7g. Ek tablo: zemin gevşekliği ve su buzu (skora girmez)

Bu iki katman benzerlik skoruna **katılmaz**; her hedef, ikiz takımındaki alanlar ve Türkiye'nin en iyi penceresi için değerler yan yana gösterilir.

| Katman | Hedef tarafı | Dünya tarafı |
| --- | --- | --- |
| Zemin gevşekliği | Mars: yörüngeden ölçülen termal eylemsizlik (yayınlanmış değerler; Jezero için THEMIS mozaiği). Ay: bu tabloda yok. | ISRIC SoilGrids (250 m, 0-5 cm): kum, kil, iri parça oranı ve hacim ağırlığı |
| Su buzu | Ay ve Mars: yayınlanmış buz bulguları | Kuzey Yarımküre permafrost haritası (Obu vd., 1 km) |

- Termal eylemsizlik ile SoilGrids farklı büyüklükleri ölçer; bu yüzden **birebir karşılaştırılmaz**. SoilGrids çıplak kaya, tuz düzlüğü ve buz alanlarında genelde değer vermez ("veri yok").
- SoilGrids sorgu sınırı nedeniyle (dakikada 5) noktalar 13 saniye arayla sorgulanır ve `ek_soilgrids.csv` dosyasına kaydedilir; ilk çalıştırmada birkaç dakika sürer.
- Permafrost haritası bir kez indirilir (`geo/` klasörüne). Güney Yarımküre'deki alanlar (Atacama) haritanın kapsamı dışındadır.

In [ ]:
import time
from pyproj import Transformer as _Tr2

# ---------- Hangi noktalar? Her hedef: ikiz takımı (7. bölüm) + Türkiye'nin en iyi penceresi (7b) ----------
pts = twin_teams[["target", "site", "lat", "lon"]].assign(rol="ikiz takımı")
tr_best = disc.sort_values("score", ascending=False).groupby("target").head(1)[["target", "site", "lat", "lon"]].assign(rol="Türkiye en iyi")
pts = pd.concat([pts, tr_best], ignore_index=True)
pts["lat"], pts["lon"] = pts["lat"].round(3), pts["lon"].round(3)
uniq = pts[["site", "lat", "lon"]].drop_duplicates().reset_index(drop=True)
print("Sorgulanacak Dünya noktası:", len(uniq))

# ---------- Dünya: SoilGrids ----------
SG_PROPS = ["sand", "clay", "cfvo", "bdod"]
def soilgrids(lat, lon):
    for attempt in range(3):
        try:
            r = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query",
                             params=[("lon", lon), ("lat", lat), ("depth", "0-5cm"), ("value", "mean")] +
                                    [("property", p) for p in SG_PROPS], timeout=60)
            if r.status_code == 429:
                time.sleep(30); continue
            out = {}
            for layer in r.json()["properties"]["layers"]:
                v = layer["depths"][0]["values"].get("mean")
                d = layer["unit_measure"]["d_factor"]
                out[layer["name"]] = None if v is None else v / d
            return out
        except Exception as err:
            last = str(err)[:60]; time.sleep(15)
    return {"hata": last}

path = "ek_soilgrids.csv"
sg = pd.read_csv(path) if (os.path.exists(path) and not FORCE) else pd.DataFrame(columns=["lat", "lon"])
have = set(zip(sg["lat"].round(3), sg["lon"].round(3)))
todo = uniq[[(a, b) not in have for a, b in zip(uniq["lat"], uniq["lon"])]]
for i, r in todo.iterrows():
    res = soilgrids(r["lat"], r["lon"])
    sg = pd.concat([sg, pd.DataFrame([{"lat": r["lat"], "lon": r["lon"], **res}])], ignore_index=True)
    sg.to_csv(path, index=False)
    print(f"  SoilGrids {r['site']} ({r['lat']}, {r['lon']}): {res}")
    time.sleep(13)
sg = sg.drop_duplicates(["lat", "lon"], keep="last")

# ---------- Dünya: permafrost (Obu vd., Kuzey Yarımküre) ----------
os.makedirs("geo", exist_ok=True)
pz = "geo/UiO_PEX_PERZONES_5.0_20181128_2000_2016_NH.zip"
if not os.path.exists(pz):
    print("Permafrost haritası indiriliyor...")
    with requests.get("https://store.pangaea.de/Publications/ObuJ-etal_2018/UiO_PEX_PERZONES_5.0_20181128_2000_2016_NH.zip",
                      stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(pz, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
if not os.path.isdir("/content/permafrost"):
    zipfile.ZipFile(pz).extractall("/content/permafrost")
pshp = glob.glob("/content/permafrost/**/*.shp", recursive=True)
print("Permafrost dosyaları:", [os.path.basename(p) for p in pshp])
pcrs = gpd.read_file(pshp[0], rows=1).crs
to_p = _Tr2.from_crs("EPSG:4326", pcrs, always_xy=True)
def permafrost(lat, lon):
    if lat < 0:
        return "kapsam dışı (Güney Yarımküre)"
    x, y = to_p.transform(lon, lat)
    g = gpd.read_file(pshp[0], bbox=(x - 2000, y - 2000, x + 2000, y + 2000))
    g = g[g.contains(Point(x, y))]
    if g.empty:
        return "yok"
    cols = [c for c in g.columns if c != "geometry"]
    return " | ".join(f"{c}={g.iloc[0][c]}" for c in cols)
uniq["permafrost"] = [permafrost(a, b) for a, b in zip(uniq["lat"], uniq["lon"])]

earth_ek = uniq.merge(sg, on=["lat", "lon"], how="left")
def zemin_txt(r):
    if pd.isna(r.get("sand")) and pd.isna(r.get("clay")):
        return "veri yok (SoilGrids bu noktada değer vermiyor)"
    f = lambda v, u: "-" if pd.isna(v) else f"{v:.0f}{u}" if u == "%" else f"{v:.2f} {u}"
    return (f"kum {f(r.get('sand'), '%')}, kil {f(r.get('clay'), '%')}, iri parça {f(r.get('cfvo'), '%')}, "
            f"hacim ağırlığı {f(r.get('bdod'), 'g/cm³')}")
earth_ek["zemin"] = earth_ek.apply(zemin_txt, axis=1)

# ---------- Hedef tarafı: yayınlanmış değerler ----------
# Jezero: THEMIS termal eylemsizlik mozaiği (Edwards & Buz 2021, Zenodo 10.5281/zenodo.4931348)
jz = "geo/Jezero_Thermal_Inertia_v1.tiff"
if not os.path.exists(jz):
    with requests.get("https://zenodo.org/records/4931348/files/Jezero_Thermal_Inertia_v1.tiff?download=1", timeout=300) as r:
        r.raise_for_status(); open(jz, "wb").write(r.content)
with rasterio.open(jz) as src:
    ti = src.read(1).astype(float)
ti = ti[(ti > 0) & np.isfinite(ti)]
jz_txt = f"mozaik ortancası {np.median(ti):.0f} (aralık p10-p90: {np.percentile(ti, 10):.0f}-{np.percentile(ti, 90):.0f})"

MARS_ICE = ("Yüzeye yakın buz Mars'ın orta enlemlerinde ve kutuplarında (NASA/JPL, 10 Aralık 2019); "
            "hedef ekvatora yakın ({lat:.1f}°) ve bu kaynakta buz bölgesi olarak geçmiyor.")
TARGET_EK = {
    "Mars_Gusev_Spirit":   ("Termal eylemsizlik 315 (TES, iniş alanı)", "Golombek vd. 2005, LPSC XXXVI, 1542"),
    "Mars_Meridiani_Oppy": ("Termal eylemsizlik 200 (TES) / 190 (THEMIS), iniş alanı", "Golombek vd. 2005, LPSC XXXVI, 1542"),
    "Mars_Gale_Curiosity": ("Termal eylemsizlik 250-410 (iniş alanının çoğu; sertleşmiş yüzey + gevşek malzeme)", "Fergason vd. 2012, Space Sci. Rev. 170"),
    "Mars_Jezero":         ("Termal eylemsizlik " + jz_txt, "Edwards & Buz 2021, Zenodo 4931348"),
}
MOON_ICE = {
    "Moon_Shackleton": ("İç duvarlarda radar verisi az miktarda buzla uyumlu; üst sınır ağırlıkça %5-10 (LRO Mini-RF)", "Astronomy, 4 Eylül 2012"),
    "Moon_deGerlache": ("Merkezinde büyük kalıcı gölge bölgesi; yüzey buzunun kraterin yaklaşık %2,3'ünü kaplıyor olabileceği tahmini", "Umansky vd., LPSC (de Gerlache)"),
    "Moon_Malapert":   ("Doğrudan buz ölçümü kaynağı bulunamadı; Artemis III aday bölgeleri kalıcı gölge bölgelerine yakın seçildi", "NASA, 19 Ağustos 2022"),
}
trows = []
for _, t in pd.concat([moon_t[["site", "lat"]], mars_best[["site", "lat"]]]).iterrows():
    if t["site"].startswith("Mars"):
        z, zk = TARGET_EK[t["site"]]
        b, bk = MARS_ICE.format(lat=abs(t["lat"])), "NASA/JPL 2019"
    else:
        z, zk = "bu tabloda yok", "-"
        b, bk = MOON_ICE[t["site"]]
    trows.append({"target": t["site"], "taraf": "hedef", "site": t["site"], "lat": None, "lon": None,
                  "zemin": z, "buz_permafrost": b, "kaynak": f"{zk}; {bk}"})

ek = pd.concat([pd.DataFrame(trows),
                pts.merge(earth_ek[["lat", "lon", "zemin", "permafrost"]], on=["lat", "lon"], how="left")
                   .rename(columns={"rol": "taraf", "permafrost": "buz_permafrost"})
                   .assign(kaynak="ISRIC SoilGrids 2.0; Obu vd. 2019 (PANGAEA)")],
               ignore_index=True)
order = {t: i for i, t in enumerate(list(moon_t["site"]) + list(mars_best["site"]))}
ek = ek.sort_values(["target", "taraf"], key=lambda s: s.map(order) if s.name == "target" else s.map({"hedef": 0, "ikiz takımı": 1, "Türkiye en iyi": 2}))
ek.to_csv("ek_tablo.csv", index=False)
for tname, g in ek.groupby("target", sort=False):
    print(f"\n=== {tname} ===")
    for _, r in g.iterrows():
        loc = "" if pd.isna(r["lat"]) else f" ({r['lat']}, {r['lon']})"
        print(f"  [{r['taraf']}] {r['site']}{loc}\n      zemin: {r['zemin']}\n      buz/permafrost: {r['buz_permafrost']}")

## 7h. Erişilebilirlik: en yakın havalimanı

Her aday için en yakın **tarifeli seferi olan** havalimanı ve **kuş uçuşu** uzaklık (karayolu değil). Veri: OurAirports açık veritabanı (günlük güncellenir).

- **Türkiye adayları için yalnızca Türkiye'deki havalimanları** hesaba katılır (ör. Iğdır'daki adaylar için en yakın havalimanı sınırın öbür tarafında çıkabiliyor; kara sınırı kapalı olabilir). Yurt dışı analoglar için en yakın havalimanı alınır.
- **Pencerede havalimanı uyarısı:** Pencere merkezine 6 km'den yakın (yani 12 km'lik pencerenin içinde) herhangi bir havalimanı ya da pist varsa "pencerede havalimanı var" notu düşülür; bu pencere doğal bir saha testi alanı sayılmamalıdır.

In [ ]:
air_all = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv",
                      usecols=["name", "type", "latitude_deg", "longitude_deg", "iso_country", "municipality", "iata_code", "scheduled_service"])
air = air_all[air_all["type"].isin(["large_airport", "medium_airport"]) & (air_all["scheduled_service"] == "yes")].reset_index(drop=True)
strips = air_all[air_all["type"].isin(["large_airport", "medium_airport", "small_airport"])].reset_index(drop=True)
TR_SITES = set(earth.loc[earth["group"].isin(["turkey", "scan"]), "site"])
print("Tarifeli havalimanı sayısı:", len(air))

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

acc_pts = pd.concat([pts[["site", "lat", "lon"]],
                     earth[earth["group"].isin(["known", "turkey"])].groupby("site")[["lat", "lon"]].mean().reset_index()],
                    ignore_index=True).drop_duplicates(["site", "lat", "lon"])
rows = []
for _, p in acc_pts.iterrows():
    a = air[air["iso_country"] == "TR"].reset_index(drop=True) if p["site"] in TR_SITES else air
    d = haversine_km(p["lat"], p["lon"], a["latitude_deg"].values, a["longitude_deg"].values)
    k = int(np.argmin(d))
    ds = haversine_km(p["lat"], p["lon"], strips["latitude_deg"].values, strips["longitude_deg"].values)
    inside = strips[ds < WIN_M / 2000]
    rows.append({"site": p["site"], "lat": round(p["lat"], 3), "lon": round(p["lon"], 3),
                 "havalimani": a.loc[k, "name"], "iata": a.loc[k, "iata_code"], "sehir": a.loc[k, "municipality"],
                 "kus_ucusu_km": round(float(d[k]), 1),
                 "pencerede_havalimani": "; ".join(f"{n} ({x:.1f} km)" for n, x in zip(inside["name"], ds[ds < WIN_M / 2000])) or ""})
access = pd.DataFrame(rows).sort_values("kus_ucusu_km")
access.to_csv("accessibility.csv", index=False)
print(access.to_string(index=False))
warn = access[access["pencerede_havalimani"] != ""]
if len(warn):
    print("\nUYARI: Penceresinde havalimanı ya da pist olan adaylar (doğal saha testi alanı sayılmamalı):")
    print(warn[["site", "lat", "lon", "pencerede_havalimani"]].to_string(index=False))

## 7i. Sağlamlık testleri

Jürinin "neden bu ağırlıklar?" sorusuna sayıyla cevap. Hepsi **genel modda**, Ay (40 m) ve Mars (200 m) için:

1. **Ağırlık hassasiyeti:** Her özelliğin ağırlığı rastgele %50 azaltılıp %50 artırılarak (0,5x-1,5x) 200 kez sıralama yeniden yapılır. Her hedef için: AUC'nin aralığı, Türkiye'nin en iyi penceresinin kaç denemede aynı kaldığı.
2. **Skor aralığı:** Aynı denemelerde Türkiye'nin en iyi penceresinin skorunun %5-%95 aralığı.
3. **Birini-dışarıda-bırak:** Bilinen analoglardan biri "bilinmiyor" sayılır (ölçeklemeye ve doğrulamaya katılmaz). Model bu alanı kendiliğinden bulabiliyor mu? Alanın en iyi penceresinin bütün pencereler arasındaki sırası (üst %) verilir.

In [ ]:
rng = np.random.default_rng(42)
N_DRAWS = 200
ROB_SETS = {f"Ay LOLA {R_MOON} m": (moon_t, cand_moon), "Mars HRSC_MOLA_200m": DATASETS["Mars HRSC_MOLA_200m"]}

# 1-2) Ağırlık hassasiyeti ve skor aralığı
sens_rows = []
for ds, (tg, c) in ROB_SETS.items():
    base = rank(c, tg, W)
    base_tr = (base[base["group"].isin(["turkey", "scan"])].sort_values("score", ascending=False)
               .groupby("target").head(1).set_index("target")[["site", "lat", "lon", "score"]])
    draws = []
    for i in range(N_DRAWS):
        w = W * rng.uniform(0.5, 1.5, len(W))
        r = rank(c, tg, w)
        v = validate(r).set_index("target")["AUC"]
        trb = r[r["group"].isin(["turkey", "scan"])].sort_values("score", ascending=False).groupby("target").head(1).set_index("target")
        for t in tg["site"]:
            same = (abs(trb.loc[t, "lat"] - base_tr.loc[t, "lat"]) < 1e-6) and (abs(trb.loc[t, "lon"] - base_tr.loc[t, "lon"]) < 1e-6)
            bscore = r[(r["target"] == t) & (np.abs(r["lat"] - base_tr.loc[t, "lat"]) < 1e-6) & (np.abs(r["lon"] - base_tr.loc[t, "lon"]) < 1e-6)]["score"].iloc[0]
            draws.append({"target": t, "AUC": v[t], "ayni_tr_en_iyi": same, "tr_en_iyi_skor": bscore})
    d = pd.DataFrame(draws)
    for t, g in d.groupby("target"):
        sens_rows.append({"veri": ds, "target": t, "AUC_genel": round(validate(base).set_index("target").loc[t, "AUC"], 2),
                          "AUC_p5": round(g["AUC"].quantile(0.05), 2), "AUC_p95": round(g["AUC"].quantile(0.95), 2),
                          "tr_en_iyi": base_tr.loc[t, "site"], "tr_en_iyi_ayni_kaldi_%": round(100 * g["ayni_tr_en_iyi"].mean()),
                          "tr_skor": round(base_tr.loc[t, "score"], 1),
                          "tr_skor_p5": round(g["tr_en_iyi_skor"].quantile(0.05), 1), "tr_skor_p95": round(g["tr_en_iyi_skor"].quantile(0.95), 1)})
sens = pd.DataFrame(sens_rows)
sens.to_csv("robustness_weights.csv", index=False)
print(f"=== Ağırlık hassasiyeti ({N_DRAWS} deneme, ağırlıklar 0,5x-1,5x) ===")
print(sens.to_string(index=False))

# 3) Birini-dışarıda-bırak
loo_rows = []
known_sites = sorted(earth.loc[earth["group"] == "known", "site"].unique())
for ds, (tg, c) in ROB_SETS.items():
    for s in known_sites:
        if s not in set(c["site"]):
            continue
        c2 = c.copy()
        c2.loc[c2["site"] == s, "group"] = "heldout"        # ölçeklemeye ve doğrulamaya katılmaz
        r = rank(c2, tg, W)
        v = validate(r).set_index("target")["AUC"]
        for t, g in r.groupby("target"):
            g = g.sort_values("score", ascending=False).reset_index(drop=True)
            others = g[(g["site"] != s)]
            best = g[g["site"] == s]["score"].max()
            pct = 100 * (others["score"] > best).mean()       # bu alanın en iyi penceresinden daha yüksek skorlu pencerelerin oranı
            loo_rows.append({"veri": ds, "disarida_birakilan": s, "target": t,
                             "ust_yuzde": round(pct, 1), "AUC_kalanlarla": round(v[t], 2)})
loo = pd.DataFrame(loo_rows)
loo.to_csv("robustness_loo.csv", index=False)
print("\n=== Birini-dışarıda-bırak: dışarıda bırakılan alanın en iyi penceresi, bütün pencerelerin üst yüzde kaçında? ===")
print("(küçük = model alanı kendiliğinden buluyor; her hedef için en iyi sonuç ve hedeflerin ortalaması)")
summ_loo = loo.groupby(["veri", "disarida_birakilan"]).agg(en_iyi_ust_yuzde=("ust_yuzde", "min"),
                                                           ortalama_ust_yuzde=("ust_yuzde", "mean"),
                                                           AUC_ortalama=("AUC_kalanlarla", "mean")).round(2)
print(summ_loo.to_string())

fig, ax = plt.subplots(1, 2, figsize=(14, 0.45 * len(sens) + 2))
lab = sens["target"] + " (" + sens["veri"].str.split().str[0] + ")"
ax[0].barh(lab, sens["tr_en_iyi_ayni_kaldi_%"], color="#3B4C8C")
ax[0].set_xlim(0, 100); ax[0].set_xlabel("Türkiye'nin en iyi penceresi aynı kaldı (% deneme)")
ax[0].set_title("Ağırlıklar ±%50 değişince")
ax[1].errorbar(sens["AUC_genel"], lab, xerr=[(sens["AUC_genel"] - sens["AUC_p5"]).clip(lower=0), (sens["AUC_p95"] - sens["AUC_genel"]).clip(lower=0)], fmt="o", color="#B4462A")
ax[1].axvline(0.5, ls="--", color="gray"); ax[1].set_xlim(0.3, 1.0); ax[1].set_xlabel("AUC (nokta: genel; çizgi: %5-%95)")
ax[1].set_title("Doğrulama ağırlıklara ne kadar duyarlı?")
plt.tight_layout(); plt.savefig("fig_robustness.png", dpi=150); plt.show()

## 7j. EMIT mineral doğrulaması (NASA, ISS)

NASA EMIT görüntüleyici spektrometresinin mineral ürünü (60 m) ile mineral notlarındaki iddialar kontrol edilir. Her alan için en az bulutlu sahne seçilir; sahnenin içinde **alan merkezine 4 km'den yakın** pikseller sayılır ve EMIT'in tanıdığı mineraller (grup 1: demirli mineraller; grup 2: karbonatlar, sülfatlar, killer vb.) listelenir.

- **Giriş:** Colab Secrets'ta `EARTHDATA_USERNAME` ve `EARTHDATA_PASSWORD` olmalı (Not defteri erişimi açık). Yoksa bu bölüm atlanır, not defterinin geri kalanı etkilenmez.
- Halit (kaya tuzu) ve bazalt bu dalga boylarında zayıf iz verir; EMIT'in bunları göstermemesi "yok" anlamına gelmez.
- Her sahne bir kez indirilir (`geo/emit/` klasörüne).

In [ ]:
!pip install -q earthaccess
import earthaccess, xarray as xr

EMIT_SITES = [("Tuz Gölü", 38.80, 33.40), ("Tuz Gölü doğusu", 38.542, 33.66), ("Salda Gölü", 37.55, 29.68), ("Kula", 38.57, 28.65)]
CHECK = ["gypsum", "anhydrite", "calcite", "dolomite", "magnesite", "hydromagnesite", "olivine", "halite", "jarosite"]
emit_rows, emit_lib = [], None
try:
    from google.colab import userdata
    os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
    os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
    auth = earthaccess.login(strategy="environment")
    EMIT_OK = bool(auth and auth.authenticated)
except Exception as err:
    EMIT_OK = False
    print("EMIT bölümü atlandı: Earthdata girişi yapılamadı →", str(err)[:120])

def emit_best_granule(lat, lon):
    best = None
    for ver in ["002", "001"]:
        try:
            res = earthaccess.search_data(short_name="EMITL2BMIN", version=ver, point=(lon, lat), count=100)
        except Exception:
            continue
        for g in res:
            cc = g["umm"].get("CloudCover")
            cc = 100 if cc is None else float(cc)
            if best is None or cc < best[0] or (cc == best[0] and ver == "002"):
                best = (cc, ver, g)
    return best

def find_var(ds, name):
    for k in list(ds.variables):
        if k.lower() == name.lower():
            return ds[k]
    raise KeyError(name)

if EMIT_OK:
    os.makedirs("geo/emit", exist_ok=True)
    for name, lat, lon in EMIT_SITES:
        b = emit_best_granule(lat, lon)
        if b is None:
            print(f"{name}: sahne bulunamadı"); continue
        cc, ver, g = b
        links = [l for l in g.data_links() if "EMIT_L2B_MIN_" in l and l.endswith(".nc") and "MINUNCERT" not in l]
        local = os.path.join("geo/emit", os.path.basename(links[0]))
        if not os.path.exists(local):
            earthaccess.download([links[0]], local_path="geo/emit")
        root = xr.open_dataset(local)
        loc = xr.open_dataset(local, group="location")
        meta = xr.open_dataset(local, group="mineral_metadata").to_dataframe().reset_index()
        meta["name"] = meta["name"].map(lambda x: x.decode() if isinstance(x, bytes) else str(x))
        if emit_lib is None:
            emit_lib = meta
        la, lo = find_var(loc, "lat").values, find_var(loc, "lon").values
        near = haversine_km(lat, lon, la, lo) < 4.0
        n_near = int(near.sum())
        out = {"alan": name, "sahne": g["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10],
               "surum": ver, "bulut_%": cc, "piksel_4km": n_near}
        for grp in [1, 2]:
            mid = root[f"group_{grp}_mineral_id"].values
            bd = root[f"group_{grp}_band_depth"].values
            ok = near & (mid > 0) & (bd > 0)
            out[f"grup{grp}_tespit_%"] = round(100 * ok.sum() / max(n_near, 1), 1)
            ids, cnt = np.unique(mid[ok], return_counts=True)
            names = meta[meta["group"] == grp].set_index("index")["name"] if "group" in meta else meta.set_index("index")["name"]
            top = sorted(zip(cnt, ids), reverse=True)[:6]
            out[f"grup{grp}_en_sik"] = "; ".join(f"{names.get(i, i)} ({c})" for c, i in top) or "-"
        emit_rows.append(out)
        print(f"\n=== {name}: {out['sahne']} (sürüm {ver}, bulut %{cc:.0f}), 4 km içinde {n_near} piksel ===")
        print(f"  Grup 1 (demirli) tespit oranı %{out['grup1_tespit_%']}: {out['grup1_en_sik']}")
        print(f"  Grup 2 (karbonat, sülfat, kil...) tespit oranı %{out['grup2_tespit_%']}: {out['grup2_en_sik']}")

    emit_df = pd.DataFrame(emit_rows)
    emit_df.to_csv("emit_minerals.csv", index=False)
    if emit_lib is not None:
        print("\nEMIT kütüphanesinde aranan mineraller (adında geçiyor mu?):")
        lib = emit_lib["name"].str.lower()
        for m in CHECK:
            hits = sorted(set(emit_lib.loc[lib.str.contains(m), "name"]))
            print(f"  {m}: {'VAR → ' + '; '.join(hits[:4]) if hits else 'kütüphanede yok'}")
        emit_lib.to_csv("emit_library.csv", index=False)

## 8b. Web sitesi verisi

Sitenin okuyacağı bütün veri `web/` klasörüne yazılır. Sitede elle yazılmış sayı yoktur; her şey buradan gelir.

- `web/data/*.json`: hedefler, bütün pencereler ve her hedef × mod için skorlar, ikiz takımları, Türkiye keşfi, tarama ızgarası, iklim takvimi, doğrulama ve sağlamlık, kayaç, ek tablo, EMIT, erişilebilirlik
- `web/img/cards/`: karşılaştırma kartlarının panelleri ayrı ayrı (yazısız; başlıklar sitede dört dilde)
- `web/img/fig/`: not defterindeki grafikler

Metinler ve etiketler kod olarak saklanır (ör. `slope_mean`, `volcanic`); dört dildeki karşılıkları sitenin kendi dil dosyalarındadır.

In [ ]:
import json, shutil, datetime
os.makedirs("web/data", exist_ok=True); os.makedirs("web/img/cards", exist_ok=True); os.makedirs("web/img/fig", exist_ok=True)
LABEL_CODE = {v: k for k, v in LABEL_TR.items()}
r1 = lambda x, n=1: None if (x is None or (isinstance(x, float) and np.isnan(x))) else round(float(x), n)

def clean(o):
    # JSON'da NaN geçersiz: boş değerleri null yap, numpy sayılarını Python sayısına çevir
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (float, np.floating)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (datetime.date, datetime.datetime, pd.Timestamp)):
        return o.isoformat()[:10]
    return o

def dump(name, obj):
    with open(f"web/data/{name}.json", "w", encoding="utf-8") as fh:
        json.dump(clean(obj), fh, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    print(f"  web/data/{name}.json  {os.path.getsize(f'web/data/{name}.json') / 1e3:.0f} KB")

# ---------- Havalimanı bilgisi her pencere için ----------
def airport_info(df):
    out = []
    for site, la, lo in zip(df["site"], df["lat"], df["lon"]):
        a = air[air["iso_country"] == "TR"] if site in TR_SITES else air
        d = haversine_km(la, lo, a["latitude_deg"].values, a["longitude_deg"].values)
        k = int(np.argmin(d))
        ds_ = haversine_km(la, lo, strips["latitude_deg"].values, strips["longitude_deg"].values)
        inside = strips[ds_ < WIN_M / 2000]
        out.append({"ap": a.iloc[k]["name"], "iata": a.iloc[k]["iata_code"] if isinstance(a.iloc[k]["iata_code"], str) else "",
                    "ap_km": r1(d[k]), "strip": "; ".join(inside["name"])})
    return out

# ---------- Hedefler ----------
rt = rock_targets.set_index("site")
targets = []
for _, t in pd.concat([moon_t.assign(body="Moon"), mars_best.assign(body="Mars")]).iterrows():
    targets.append({"id": t["site"], "body": t["body"], "lat": r1(t["lat"], 3), "lon": r1(t["lon"], 3), "res_m": r1(t["res_m"]),
                    "rock": rt.loc[t["site"], "rock_family"], "rock_map": rock_targets.set_index("site").loc[t["site"]].get("unit"),
                    "rock_override": bool(rt.loc[t["site"], "duzeltme"]),
                    "features": {f: r1(t[f], 3) for f in F}})
DS_KEYS = {f"Ay LOLA {R_MOON} m": "moon40", "Mars HRSC_MOLA_200m": "mars200", "Mars MOLA_463m": "mars463"}
dump("meta", {"generated": datetime.date.today().isoformat(),
              "scan": {"cell_deg": 0.1, "min_barren": SCAN_MIN_BARREN, "max_sites": SCAN_MAX_SITES, "min_sep_deg": SCAN_MIN_SEP}, "targets": targets,
              "modes": list(MODES), "weights": {m: {k: r1(v, 2) for k, v in w.items()} for m, w in MODES.items()},
              "datasets": {v: k for k, v in DS_KEYS.items()}, "features": F + ["w_barren_frac", "rock_match"]})

# ---------- Pencereler ve skorlar (her veri seti) ----------
for ds, (tg, c) in DATASETS.items():
    key = DS_KEYS[ds]
    c = c.reset_index(drop=True)
    wins = c[["site", "group", "lat", "lon"]].copy()
    wins["rock"] = c["rock_family"].where(c["rock_family"].notna(), None) if "rock_family" in c else None
    feats = {f: [r1(v, 3) for v in c[f]] for f in F + ["w_barren_frac"]}
    ap = airport_info(c)
    scores = {}
    for m in MODES:
        r = mode_ranks[(ds, m)]
        for t, g in r.groupby("target"):
            g = g.reset_index(drop=True)                     # rank() sırayı korur: g satırları c ile aynı sırada
            scores.setdefault(t, {})[m] = [r1(v) for v in g["score"]]
    dump(f"windows_{key}", {"site": list(wins["site"]), "group": list(wins["group"]),
                            "lat": [r1(v, 4) for v in wins["lat"]], "lon": [r1(v, 4) for v in wins["lon"]],
                            "rock": list(wins["rock"]), "f": feats,
                            "ap": [a["ap"] for a in ap], "iata": [a["iata"] for a in ap], "ap_km": [a["ap_km"] for a in ap],
                            "strip": [a["strip"] for a in ap], "scores": scores})

# ---------- İkiz takımları (genel ve modlar; yapısal) ----------
twins = {}
for ds in TWIN_SETS:
    tg, c = DATASETS[ds]
    for _, t in tg.iterrows():
        entry = {}
        for m, w in MODES.items():
            wp = w if m == "genel" else w[PRIORITY[m] + ["w_barren_frac"]]
            wp = wp.drop("rock_match", errors="ignore") if m == "genel" else wp
            c_t, t_t = with_rock(c, t) if "rock_match" in wp.index else (c, t)
            team, miss, cov = twin_team(c_t, t_t, w=wp)
            entry[m] = {"coverage": r1(cov, 0), "missing": [LABEL_CODE.get(x, x) for x in miss],
                        "priority": list(wp.index),
                        "members": [{"site": x["site"], "group": x["group"], "lat": x["lat"], "lon": x["lon"],
                                     "covers": [LABEL_CODE.get(y.strip(), y.strip()) for y in x["karsiladigi"].split(",") if y.strip()]}
                                    for x in team]}
        twins[t["site"]] = {"dataset": DS_KEYS[ds], **entry}
dump("twins", twins)

# ---------- Türkiye keşfi, tarama ızgarası ----------
dump("turkey", [{"target": r["target"], "site": r["site"], "group": r["group"], "lat": r1(r["lat"], 4), "lon": r1(r["lon"], 4),
                 "score": r1(r["score"]), "known_best": r1(r["bilinen_en_iyi"]), "ratio": r1(r["bilinen_en_iyiye_oran_%"], 0)}
                for _, r in disc.iterrows()])
sg_ = pd.read_csv("scan_grid_turkey.csv")
dump("scan_grid", {"lat": [r1(v, 2) for v in sg_["lat"]], "lon": [r1(v, 2) for v in sg_["lon"]],
                   "barren": [r1(v, 2) for v in sg_["barren"]]})

# ---------- İklim takvimi (bütün alanlar) ----------
clim = {}
for _, r in clim_m.iterrows():
    status, best, T, P, S, relaxed = calendar(r)
    clim[r["site"]] = {"T": [r1(v) for v in T], "P": [r1(v, 2) for v in P], "S": [r1(v, 1) for v in S],
                       "status": [int(v) for v in status], "best": best, "relaxed": bool(relaxed)}
dump("climate", clim)

# ---------- Doğrulama ve sağlamlık ----------
MODE_CODE = {v: k for k, v in MODE_TR.items()}
LAYER_CODE = {"arazi + çoraklık": "terrain_barren", "sadece arazi": "terrain"}
fix = lambda d: d.assign(**({"veri": d["veri"].map(DS_KEYS)} if "veri" in d else {}))
dump("validation", {"base": fix(val).assign(katmanlar=val["katmanlar"].map(LAYER_CODE)).to_dict(orient="records"),
                    "modes": fix(summ).assign(mod=summ["mod"].map(MODE_CODE)).drop(columns=["uyari"]).to_dict(orient="records"),
                    "weights": fix(sens).to_dict(orient="records"),
                    "loo": fix(summ_loo.reset_index()).to_dict(orient="records")})

# ---------- Kayaç, ek tablo, EMIT, erişilebilirlik ----------
FAM_CODE = {v: k for k, v in FAM_TR.items() if k is not None}
dump("rock", {"targets": rock_targets[["site", "unit", "unit_desc", "rock_family", "tam_icinde"]]
                          .assign(map_family=rock_targets["harita_aile"].map(FAM_CODE)).to_dict(orient="records"),
              "earth_sites": rock_earth[rock_earth["group"] != "scan"].groupby("site")["rock_family"]
                          .agg(lambda s: s.fillna("unknown").value_counts().to_dict()).to_dict()})
ek_num = uniq.merge(sg, on=["lat", "lon"], how="left")
dump("extra", {"points": pts.to_dict(orient="records"),
               "earth": [{"site": r["site"], "lat": r["lat"], "lon": r["lon"],
                          "sand": r1(r.get("sand")), "clay": r1(r.get("clay")), "cfvo": r1(r.get("cfvo")), "bdod": r1(r.get("bdod"), 2),
                          "permafrost": (re.search(r"EXTENT=(\w+)", str(r["permafrost"])) or [None, None])[1]
                                        if "EXTENT=" in str(r["permafrost"]) else ("outside" if "kapsam" in str(r["permafrost"]) else "none")}
                         for _, r in ek_num.iterrows()],
               "mars_ti": {"Mars_Gusev_Spirit": "315", "Mars_Meridiani_Oppy": "200 / 190",
                           "Mars_Gale_Curiosity": "250-410", "Mars_Jezero": f"{np.median(ti):.0f} ({np.percentile(ti, 10):.0f}-{np.percentile(ti, 90):.0f})"}})
if "emit_df" in globals() and len(emit_df):
    EMIT_ID = {"Tuz Gölü": "tuzgolu", "Tuz Gölü doğusu": "tuzgolu_east", "Salda Gölü": "salda", "Kula": "kula"}
    dump("emit", emit_df.assign(id=emit_df["alan"].map(EMIT_ID)).to_dict(orient="records"))
dump("access", access.fillna("").to_dict(orient="records"))

# ---------- Görseller: kart panelleri (yazısız) ve grafikler ----------
from PIL import Image
def save_img(arr, path, cmap=None, vmin=0, vmax=1):
    if arr is None:
        return False
    if cmap is not None:
        plt.imsave(path, arr, cmap=cmap, vmin=vmin, vmax=vmax)
    else:
        a = np.asarray(arr)
        if a.dtype != np.uint8:
            a = a.astype(float)
            a = a / 255.0 if np.nanmax(a) > 1.5 else a
            a = (np.clip(np.nan_to_num(a), 0, 1) * 255).astype(np.uint8)
        Image.fromarray(a).save(path)
    im = Image.open(path).convert("RGB")
    im.thumbnail((600, 600)); im.save(path.replace(".png", ".jpg"), quality=85); os.remove(path)
    return True
card_meta = []
for c in cards:
    base = f"web/img/cards/{c['target']}"
    have = {"target": save_img(c.get("t_hs"), base + "_target.png", "gray"),
            "earth": save_img(c.get("e_hs"), base + "_earth.png", "gray"),
            "rgb": save_img(c.get("rgb"), base + "_rgb.png"),
            "wc": save_img(c.get("wc"), base + "_wc.png", wc_cmap, 0, 3)}
    card_meta.append({"target": c["target"], "site": c["site"], "lat": r1(c["lat"], 4), "lon": r1(c["lon"], 4),
                      "score": r1(c["score"]), "ex": c.get("ex", 1), "res_t": r1(c.get("res_t")),
                      "rgb_date": c.get("rgb_date"), "rgb_cloud": r1(c.get("rgb_cloud"), 2), "img": have})
dump("cards", card_meta)

# ---------- Ülke sınırları (Natural Earth 1:50m, kamu malı): Türkiye haritasında sınır çizgisi ve çevrimdışı yedek harita ----------
try:
    import geopandas as gpd
    ne = None
    for u in ["https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson",
              "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip"]:
        try:
            ne = gpd.read_file(u)
            break
        except Exception as err:
            print("Sınır verisi okunamadı:", u, err)
    iso = "ISO_A3" if "ISO_A3" in ne.columns else "iso_a3"
    ne = ne[[iso, "geometry"]].rename(columns={iso: "iso"})
    ne["geometry"] = ne.geometry.simplify(0.02, preserve_topology=True)   # ~2 km: site için yeterli, dosya küçük kalır
    gj = json.loads(ne.to_json(drop_id=True))
    def _rnd(c):
        return [_rnd(x) for x in c] if isinstance(c[0], (list, tuple)) else [round(c[0], 3), round(c[1], 3)]
    for f in gj["features"]:
        f["geometry"]["coordinates"] = _rnd(f["geometry"]["coordinates"])
    dump("countries", gj)
except Exception as err:
    print("countries.json atlandı (site onsuz da çalışır):", err)
for f in ["fig_turkey_scan.png", "fig_robustness.png", "fig_modes.png"]:
    if os.path.exists(f):
        shutil.copy(f, "web/img/fig/" + f)
print("\nweb/ klasörü hazır:", sum(len(fs) for _, _, fs in os.walk("web")), "dosya")

## 8c. Rover simülasyonu için yükseklik ızgaraları

Sitedeki rover simülasyonu, her hedef için üç araziyi aynı çözünürlükte karşılaştırır: **hedefin kendisi**, **Rover modunda Türkiye'nin en iyi penceresi** ve **Rover modunda dünyanın en iyi penceresi**. Bu hücre bu 12 km'lik pencerelerin yükseklik ızgaralarını küçük dosyalar olarak `web/data/rover/` klasörüne yazar. Hesap yapmaz; rota bulma tarayıcıda çalışır. Yükseklikler 0,1 m hassasiyetle 16 bitlik tamsayı olarak saklanır.

In [ ]:
import base64

def earth_grid(site, lat, lon, res):
    # Dünya penceresini, Bölüm 4-5'teki işlemle aynı UTM ızgarasından (res metre) keser
    center = {s[0]: (s[1], s[2]) for s in SITES}[site]
    path, _ = download(site, *center)
    with rasterio.open(to_utm(path, *center, res)) as src:
        x, y = warp_xy("EPSG:4326", src.crs, [lon], [lat])
        r_ = src.res[0]
        n = int(WIN_M / r_)
        col, row = ~src.transform * (x[0], y[0])
        i0, j0 = int(round(row - n / 2)), int(round(col - n / 2))
        z = src.read(1, window=((i0, i0 + n), (j0, j0 + n)), boundless=True, fill_value=np.nan).astype(float)
    z = np.where(np.isnan(z), np.nanmean(z), z)
    return z, r_

def pack(z, res, label, site, lat, lon):
    z0 = float(np.floor(z.min()))
    q = np.round((z - z0) * 10)
    scale = 10.0
    if q.max() > 32767:                              # çok yüksek rölyef: 1 m hassasiyet
        q, scale = np.round(z - z0), 1.0
    q = q.astype("<i2")
    return {"label": label, "site": site, "lat": round(float(lat), 4), "lon": round(float(lon), 4),
            "res_m": round(float(res), 2), "n": int(z.shape[0]), "m": int(z.shape[1]),
            "z0": z0, "scale": scale, "data": base64.b64encode(q.tobytes()).decode()}

os.makedirs("web/data/rover", exist_ok=True)
mb = pd.read_csv("modes_best_windows.csv")
index = []
JOBS = ([(n, moon_wins, "Ay LOLA 40 m", 40) for n, _, _ in MOON_TARGETS] +
        [(n, mars_wins, "Mars HRSC_MOLA_200m", 200) for n, _, _ in MARS_TARGETS])
for tname, wins, ds, res in JOBS:
    try:
        zt = wins[tname][0] if tname in wins else wins[("HRSC_MOLA_200m", tname)][0]
        tlat, tlon = [(la, lo) for n, la, lo in MOON_TARGETS + MARS_TARGETS if n == tname][0]
        rt = WIN_M / zt.shape[0]
        row = mb[(mb["veri"] == ds) & (mb["target"] == tname) & (mb["mod"] == "Rover")].iloc[0]
        terrains = [pack(zt, rt, "target", tname, tlat, tlon)]
        for lab, s_, la, lo, sc in [("turkey", row["tr_en_iyi_alan"], row["tr_lat"], row["tr_lon"], row["tr_skor"]),
                                    ("world", row["en_iyi_alan"], row["en_iyi_lat"], row["en_iyi_lon"], row["en_iyi_skor"])]:
            if lab == "world" and s_ == row["tr_en_iyi_alan"]:
                continue                                 # dünyanın en iyisi zaten Türkiye'deyse tekrar etme
            ze, re_ = earth_grid(s_, la, lo, res)
            terrains.append({**pack(ze, re_, lab, s_, la, lo), "score": round(float(sc), 1)})
        dump(f"rover/{tname}", {"target": tname, "terrains": terrains})
        index.append({"target": tname, "terrains": [{k: t[k] for k in ("label", "site", "res_m", "n")} for t in terrains]})
    except Exception as err:
        print(f"✗ {tname}: {err}")
dump("rover/index", index)

## 8. Kaydedilen dosyalar

`MyDrive/MirrorTerrain` klasöründe:

| Dosya | İçerik |
| --- | --- |
| `features_earth.csv` | Dünya pencereleri, 30 m: arazi, iklim, çoraklık |
| `features_earth_40m.csv`, `features_earth_200m.csv`, `features_earth_462m.csv` | Aynı pencereler hedef çözünürlüklerinde |
| `features_targets_moon.csv`, `features_targets_mars.csv` | Ay ve Mars hedeflerinin özellikleri |
| `ranks_moon.csv`, `ranks_mars_*.csv` | Her hedef için tüm Dünya pencerelerinin benzerlik skorları |
| `validation.csv` | Doğrulama ölçütleri |
| `twin_teams.csv` | Her hedef için ikiz takımı ve Dünya'da karşılanamayan özellikler |
| `scan_grid_turkey.csv` | Türkiye taraması: her 0,1°'lik hücrenin çoraklığı |
| `turkey_discovery.csv` | Her hedef için Türkiye'nin en iyi pencereleri ve Google Haritalar linkleri |
| `fig_turkey_scan.png` | Türkiye çoraklık haritası ve tarama adayları |
| `climate_monthly.csv`, `best_months.csv`, `fig_best_month.png` | Aylık iklim, en iyi test ayı tablosu ve takvim görseli |
| `fig_candidate_cards.png`, `candidate_cards.csv` | Hedef ↔ Türkiye adayı ↔ Sentinel-2 ↔ WorldCover karşılaştırma kartları |
| `rock_earth_v2.csv`, `rock_targets.csv` | Pencerelerin ve hedeflerin kayaç aileleri |
| `ek_tablo.csv`, `ek_soilgrids.csv` | Ek tablo: zemin gevşekliği ve su buzu (skora girmez) |
| `accessibility.csv` | En yakın tarifeli havalimanı ve kuş uçuşu uzaklık |
| `robustness_weights.csv`, `robustness_loo.csv`, `fig_robustness.png` | Sağlamlık testleri |
| `emit_minerals.csv`, `emit_library.csv` | EMIT mineral doğrulaması |
| `modes_validation.csv`, `modes_validation_summary.csv` | Her test amacı modu için doğrulama ve genel moda göre AUC farkı |
| `modes_best_windows.csv` | Her hedef × mod için en iyi dünya ve Türkiye penceresi |
| `modes_twin_teams.csv`, `fig_modes.png` | Mod ikiz takımları ve modlara göre Türkiye skorları |
| `climate_earth.csv` | NASA POWER aylık iklim (en iyi test zamanı katmanı için) |
| `fig_moon_targets.png`, `fig_mars_targets_200m.png` | Ay ve Mars hedeflerinin görselleri |
| `web/data/rover/*.json` | Rover simülasyonu için hedef, Türkiye ve dünya pencerelerinin yükseklik ızgaraları |

**Sonraki adım:** web sitesi.

## 9. Sonuçları bilgisayarına indir

Bütün güncel sonuç tablolarını (CSV), grafikleri (küçültülmüş JPG) ve web sitesi verisini (`web/`) tek bir sıkıştırılmış zip dosyasında toplar. Eski sürümlerden kalan ve artık kullanılmayan dosyalar (`features_targets.csv`, `ranks_mars.csv`, `rock_earth.csv`) dahil edilmez. Tarayıcı izin isterse "İzin ver" de.

In [ ]:
import glob, zipfile, io
from PIL import Image
from google.colab import files

OBSOLETE = {"features_targets.csv", "ranks_mars.csv", "rock_earth.csv"}
zip_name = "MirrorTerrain_sonuclar.zip"
n = 0
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    for f in sorted(glob.glob("*.csv")):
        if f not in OBSOLETE:
            zf.write(f); n += 1
    for f in glob.glob("web/**/*.*", recursive=True):
        zf.write(f); n += 1
    for f in sorted(glob.glob("fig_*.png")):
        im = Image.open(f).convert("RGB"); im.thumbnail((1600, 1600))
        buf = io.BytesIO(); im.save(buf, "JPEG", quality=80, optimize=True)
        zf.writestr(f.replace(".png", ".jpg"), buf.getvalue()); n += 1
print(f"{n} dosya → {zip_name} ({os.path.getsize(zip_name) / 1e6:.1f} MB)")
files.download(zip_name)